***Indexing Laue pattern of several images using multiprocessing***

on jupyter-slurm.esrf.fr (jupyterlab or jupyterhub)

or

locally on desktop computer (with some cpus) and with access to the LaueTools environment or folder

**IN DEV**

**This Notebook is a part of the LaueTools Package** 
Author: J.-S. Micha

Last Revision:   October 2025

tested with python 3.12  on jupyterlab and jupyter hub, jupyter-notebook

**Objectives**

- 1 image

-- Index from scratch and Refine Strain

-- Check orientation and Refine Strain (followed or not by indexing from scratch), use previous results (1 or n orientation matrix)

-- [TO COME] Handle multiphase (substrate case) or multigrains indexing

- N images

-- Use previous results (DB orientation matrix, propagation)

-- [TO COME] Select subset of sample point

-- [TO COME] Track, Select, Blacklist Spots

The main goal of indexing is to:

- identify hkl Miller indices of Laue spots
- identify phase material of Laue spots (grain)

As a results:

- .fit files are generated (all indexed spots properties list )
- ####_g0.fit, ####_g1.fit, ... generated (indexed spots properties list per grain)


# SET runtime jupyterlab or not

In [ ]:
JUPYTER_LAB = True # False for jupyter hub or notebook
ON_LINUX = True
# just to have a trace of current dependencies version if problems
# if ON_LINUX:
#     !jupyter --version

# IMPORTS

In [ ]:
import multiprocessing
nbmaxcpus = multiprocessing.cpu_count()
print(f'nb of cpus available: {nbmaxcpus}')

In [ ]:
# if logged in jupyter-slurm.esrf.fr, please select the kernel LAUETOOLSENV2
# in Kernel top bar menu  /Kernel/ change Kernel

In [ ]:
# # use Lauetools installed in an ESRF conda environnement is recommended
# import os, sys
# import pickle
# import datetime
# import LaueTools as LT
# LaueToolsCode_Folder = os.path.split(LT.__file__)[0]
# print('absolute path of LaueTools package',LaueToolsCode_Folder)
# print("\nIf you are on jupyter-slurm, you should see: <module 'LaueTools' from '.../...jupyter-slurm/2025.04.3/envs/lib/python3.8/site-packages/LaueTools/__init__.py'>")
# LT

In [ ]:
#OR use your own lauetools.
# Setting absolute path to LaueTools Modules if Lauetools has not been installed with pip
if 1:
    import sys
    # for slurm machines
    # sys.path.insert(0,'/home/esrf/micha/lauetools_devNotebooks/lauetools')
    # # for lbm32gpu1 machine (test temp)
    #sys.path.insert(0,'/data/bm32/inhouse/STAFF/JSM/lauetools_devNotebooks/lauetools')

    sys.path.insert(0,'/data/bm32/inhouse/lauetoolsenv2/lib/python3.12/site-packages')
    
    import LaueTools as LT
    print('code from', LT.__file__)

In [ ]:
if not JUPYTER_LAB: # jupyterhub
    %matplotlib notebook  
else:
    %matplotlib widget 

import time,copy,os

# Third party modules
import matplotlib as mpl     # graphs and plots
import matplotlib.pyplot as plt
import numpy as np
import numpy.ma as ma

from tqdm import tqdm, tqdm_notebook
import multiprocessing
from multiprocessing import active_children, cpu_count
os.environ["OMP_NUM_THREADS"] = "1"   # 1 or 2 or 4

import itertools

from pathlib import Path

import warnings
#warnings.filterwarnings('always')

# LaueTools modules
import LaueTools.readmccd as RMCCD
#import LaueTools.LaueGeometry as F2TC
import LaueTools.indexingSpotsSet as ISS
import LaueTools.indexingAnglesLUT as IAL
import LaueTools.IOLaueTools as IOLT
import LaueTools.generaltools as GT
import LaueTools.dict_LaueTools as DictLT

# SET user scan measurements inputs

**folder of .cor files, 2D map dimensions, Laue detector type ... **

In [ ]:
# use default library of materials
mymaterials = None

## Zn dendrites

In [ ]:
INDEX_COR_FILES_IN_FOLDER = True

if 0: #INDEX_COR_FILES_IN_FOLDER:
    
    # parent folder of all dataset data (from .cor files coming from peaksearch results on images)
    maindatafolder = '/data/projects/zndendrites/ma6034/bm32/20240625/RAW_DATA'
    # subfolder where images .tiff are
    subfolderscandata = 'sample1_test/sample1_test_sample5-30m-new2-coarse-large/scan0001'
    
    datfilefolder = Path(maindatafolder+'/'+subfolderscandata+'//'+'datfiles_notebook')
    corfilefolder = datfilefolder
    
    prefixfilename= 'img_'
    suffix='.cor'
    CCDLabel = 'sCMOS'
    sizeofzeropadding = 4  # means myprefix_0023.tif  myprefix_9253.tif  myprefix_65253.tif
    
    # calibration parameters file
    detfile = Path(maindatafolder+'/'+'sample1_test/sample1_test_Ge_20240628/scan0001/Ge111calib_28june2024.det')
    
    # map dimensions
    # they can be found from BLISS command
    # dmesh motor1 value value nbsteps1 motor2 value value nbsteps2 exposure_time
    # so mapdims = (nbsteps motor2 + 1, nbsteps motor1 + 1) = (slow motor, fast motor)
    mapdims = (41,41)
    invmapdims = mapdims[1], mapdims[0]
    
    if not Path(maindatafolder+'/'+subfolderscandata).exists():
        raise ValueError(f'The folder {Path(maindatafolder+"/"+subfolderscandata)} does not exists!')
    
    # -----------------automatically generated folder ---------------------------------
    imagefolder = Path(maindatafolder+'/'+subfolderscandata)

    fitfilefolder = Path(maindatafolder+'/'+subfolderscandata+'//'+'fitfiles_notebook')
    
    if not fitfilefolder.exists():
        print('Creating "fitfilefolder" ...')
        Path.mkdir(fitfilefolder)
            
    #outputfolder = '/data/visitor/blc14894/bm32/20231007/NOBACKUP/'
    
    print('images folder:  \n    =====> %s'%imagefolder)
    print('.fit files will be written in this folder: \n    =====> %s'%fitfilefolder)

### Some orientation matrices

In [ ]:
# Guessed UB matrix (example)
UB_1 = np.array([[ 0.800890339,  0.111020812,  0.588618058],
       [ 0.006589284, -0.983840123,  0.176252946],
       [ 0.59871938 , -0.137579654, -0.788529267]])

UB_2 = np.array([[-0.944886962989832, -0.16481648329929 ,  0.282364299451425],
       [ 0.325288903402363, -0.401682702128187,  0.855560084566674],
       [-0.027898445634874,  0.899571897959609,  0.434481129960923]])

UB_3 = np.array([[ 0.798694755,  0.111538191,  0.591397078],
       [ 0.005701942, -0.983734667,  0.177867478],
       [ 0.601654241, -0.138824412, -0.786302534]])

UB_4 = np.array([[-0.944887005, -0.164816017,  0.282364339],
       [ 0.325288815, -0.401682583,  0.855559726],
       [-0.027898058,  0.899571114,  0.434480798]])

UB_5 = np.array([[ 0.790931358,  0.04942662 ,  0.610343974],
       [ 0.006278407, -0.99719131 ,  0.072531087],
       [ 0.611965387, -0.053681208, -0.788581226]])

UB_6 = np.array([[ 0.790951967,  0.049463864,  0.610128265],
       [ 0.006315838, -0.997021885,  0.072650861],
       [ 0.611938365, -0.053678736, -0.788500541]])

UB547 = np.array([[-0.944920932918629, -0.164116940216228,  0.28227572173489 ],
       [ 0.325197212589104, -0.401535417654801,  0.855019004360563],
       [-0.027816725726933,  0.89905990210913 ,  0.434389207372524]])

UB218 = np.array([[-0.496826852073101, -0.638273298777973, -0.588555208999875],
       [ 0.848892814668109, -0.497871277128914, -0.176531037918834],
       [-0.180000922484652, -0.587384245192303,  0.788803460478765]])

## Zr polycrystals

In [ ]:
#pa = '/data/visitor/a321216/bm32/20251008/RAW_DATA/ech15_R1/ech15_R1_Z1_2dmap_50x60um_1umstep_GOOD_0002/scan0001/

In [ ]:
# A321216
if 1:
    expId = 'a321216'
    # parent folder of all dataset data (from .cor files coming from peaksearch results on images)
    maindatafolder = '/data/visitor/a321216/bm32/20251008/RAW_DATA'

    # subfolder where images .tiff are
    subfolderscandata = 'ech15_R1/ech15_R1_Z1_2dmap_50x60um_1umstep/scan0001'
    subfolderscandata = 'ech15_R1/ech15_R1_Z1_2dmap_50x60um_1umstep_GOOD_0002/scan0001'
    
    prefixfilename= 'img_'
    suffix='.cor'
    CCDLabel = 'sCMOS_4M'
    sizeofzeropadding = 4  # means myprefix_0023.tif  myprefix_9253.tif  myprefix_65253.tif
    
    
    # calibration parameters file
    detfile = str(Path(maindatafolder+'/'+'Ge1/Ge1_calib_std/scan0001/Ge1.det'))
    
    # map dimensions
    # they can be found from BLISS command
    # dmesh motor1 value value nbsteps1 motor2 value value nbsteps2 exposure_time
    # so mapdims = (nbsteps motor1 + 1, nbsteps motor2 + 1) = (fast motor, slow motor)
    mapdims = (51,61)   # fast   , slow     here yech , xech
    invmapdims = mapdims[1], mapdims[0]
    
    HDF5_LOGFILE_EXISTS = True
    if HDF5_LOGFILE_EXISTS:
        pathHDF5 = os.path.join(maindatafolder,'a321216_bm32.h5')


    # -----------------automatically generated folder ---------------------------------
    imagefolder = Path(maindatafolder+'/'+subfolderscandata)
    
    mainprocessed_data = '/data/visitor/a321216/bm32/20251008/PROCESSED_DATA'
    studysubfolder = subfolderscandata
    workdir = str(Path(mainprocessed_data+'//'+studysubfolder))
    #datfilefolder = str(Path(mainprocessed_data+'/'+subfolderscandata+'//'+'corfiles'))
    datfilefolder = str(Path(maindatafolder+'/'+subfolderscandata+'//'+'img_CORfiles'))
    corfilefolder = datfilefolder

    # fitfilefolder = dirnameout_fitfile  # where results could be read (previous results)

    #dirnameout_fitfile = str(Path(workdir+'//'+f'fitfiles_{analysislabel}')) # where results are written
    fitfilefolder = Path(mainprocessed_data+'/'+subfolderscandata+'//'+'fitfiles_notebook')

In [ ]:
datfilefolder

In [ ]:
# BLC15488
if 0:
    expId = 'blc15488'
    # parent folder of all dataset data (from .cor files coming from peaksearch results on images)
    maindatafolder = '/data/projects/mapgrainxl/blc15488/bm32/20240601/RAW_DATA'

    # subfolder where images .tiff are
    subfolderscandata = 'ech15/ech15_map2Dexpo0p1sec/scan0001'

    prefixfilename= 'img_'
    suffix='.cor'
    CCDLabel = 'sCMOS'
    sizeofzeropadding = 4  # means myprefix_0023.tif  myprefix_9253.tif  myprefix_65253.tif
    
    
    # calibration parameters file
    detfile = Path(maindatafolder+'/'+'ech15/ech15_Ge/scan0001/calibGe.det')
    
    # map dimensions
    # they can be found from BLISS command
    # dmesh motor1 value value nbsteps1 motor2 value value nbsteps2 exposure_time
    # so mapdims = (nbsteps motor1 + 1, nbsteps motor2 + 1) = (fast motor, slow motor)
    mapdims = (61,51)   # fast   , slow     here xech , yech
    invmapdims = mapdims[1], mapdims[0]
    
    HDF5_LOGFILE_EXISTS = True
    if HDF5_LOGFILE_EXISTS:
        pathHDF5 = os.path.join(maindatafolder,'blc15488_bm32.h5')
        
    # -----------------automatically generated folder ---------------------------------
    # -----------------automatically generated folder ---------------------------------
    imagefolder = Path(maindatafolder+'/'+subfolderscandata)
    datfilefolder = Path(maindatafolder+'/'+subfolderscandata+'//'+'datfiles_notebook')
    #corfilefolder = datfilefolder
    corfilefolder = Path(maindatafolder+'/'+subfolderscandata+'//'+'img_CORfiles')
    
    fitfilefolder = Path(maindatafolder+'/'+subfolderscandata+'//'+'fitfiles_JSM')

    mainprocessed_data = '/data/projects/mapgrainxl/blc15488/bm32/20240601/PROCESSED_DATA/'
    studysubfolder = subfolderscandata
    #/data/projects/mapgrainxl/blc15488/bm32/20240601/PROCESSED_DATA/ech15/ech15_map2Dexpo0p1sec/scan0001/
    workdir = str(Path(mainprocessed_data+'//'+studysubfolder))
    #datfilefolder = str(Path(mainprocessed_data+'/'+subfolderscandata+'//'+'corfiles'))
    datfilefolder = str(Path(maindatafolder+'/'+subfolderscandata+'//'+'img_CORfiles'))
    corfilefolder = datfilefolder

    fitfilefolder = Path(maindatafolder+'/'+subfolderscandata+'//'+'fitfiles_JSM')

In [ ]:
os.makedirs(fitfilefolder, exist_ok=True)
#Disable if calling cor repo manually
#corfilefolder = datfilefolder

#outputfolder = '/data/visitor/blc14894/bm32/20231007/NOBACKUP/'

print('images folder:  \n    =====> %s'%imagefolder)
print('.fit files will be written in this folder: \n    =====> %s'%fitfilefolder)

### Some orientation matrices

In [ ]:
 #GOI 1 - Zr
#NB: CAUTION 
UB_Zr_1 = np.array([[-0.715649128,  0.083196626, -0.693582905],
                [ -0.274818874, -0.946425938,  0.169636022],
                [ -0.642010804,  0.310566961,  0.701280083]])

#GOI 3 - Zr
UB_Zr_3 = np.array([[ 0.448549611,  0.103010639, -0.887987377],
                    [ 0.212069947,  0.953211693,  0.217324011],
                    [ 0.868557547, -0.284717106,  0.406654304]])


# GOI - Cr = CAUTION: We were able to index a Cr grain
UB_Cr_1 = np.array([[ 0.253734688, -0.966571946, -0.036701339],
                    [ -0.151499426, -0.001523219, -0.98836153],
                    [ 0.955351885,  0.258303731, -0.148071266]])

UB_Zr_828 = np.array([[ 0.286398422302201, -0.662359423093821, -0.69545143910691 ],
                   [ 0.956705409395281,  0.235980441180103,  0.170780837853894],
                   [ 0.051855826535398, -0.712277801417387,  0.700105128281243]])

GT.getRotationAngleFrom2Matrices(UB_Zr_1, UB_Zr_828)


# ZrO2 tetragonal

In [ ]:
from pathlib import Path
if 0:
    HDF5_LOGFILE_EXISTS=False
    # parent folder of all dataset data (from .cor files coming from peaksearch results on images)
    maindatafolder = '/data/visitor/a321206/bm32/20250603/RAW_DATA'
    #/data/visitor/a321206/bm32/20250603/RAW_DATA/ZrO2/ZrO2_ZrO2_1330C/scan0002
    # subfolder where images .tiff are
    subfolderscandata = 'ZrO2/ZrO2_ZrO2_1330C/scan0002'
    
    prefixfilename= 'img_'
    suffix='.cor'
    CCDLabel = 'sCMOS'
    sizeofzeropadding = 4  # means myprefix_0023.tif  myprefix_9253.tif  myprefix_65253.tif
    
    
    # calibration parameters file
    detfile = Path(maindatafolder+'/'+'ZrO2/ZrO2_Ge/scan0001/calibGe001_A321206_mardi.det')
    
    # map dimensions
    # they can be found from BLISS command
    # dmesh motor1 value value nbsteps1 motor2 value value nbsteps2 exposure_time
    # so mapdims = (nbsteps motor1 + 1, nbsteps motor2 + 1) = (fast motor, slow motor)
    
    mapdims = (301,100)
    invmapdims = mapdims[1], mapdims[0]
    
    HDF5_LOGFILE_EXISTS = False
    if HDF5_LOGFILE_EXISTS:
        pathHDF5 = os.path.join(maindatafolder,'a321206_bm32.h5')


    # -----------------automatically generated folder ---------------------------------
    imagefolder = Path(maindatafolder+'/'+subfolderscandata)
    datfilefolder = Path(maindatafolder+'/'+subfolderscandata+'//'+'datfiles_notebook')
    #corfilefolder = datfilefolder
    corfilefolder = datfilefolder #Path(maindatafolder+'/'+subfolderscandata+'//'+'img_CORfiles')
    
    fitfilefolder = Path(maindatafolder+'/'+subfolderscandata+'//'+'fitfiles_JSM')
    
    if not fitfilefolder.exists():
        print('Creating "fitfilefolder" ...')
        Path.mkdir(fitfilefolder)
        
    #Disable if calling cor repo manually
    #corfilefolder = datfilefolder
    
    #outputfolder = '/data/visitor/blc14894/bm32/20231007/NOBACKUP/'
    

    
    print('images folder:  \n    =====> %s'%imagefolder)
    print('.fit files will be written in this folder: \n    =====> %s'%fitfilefolder)

   
    mymaterials = None


In [ ]:
if HDF5_LOGFILE_EXISTS:
    import LaueTools.logfile_reader as iohdf5
    import pandas as pd
    print('pathHDF5',pathHDF5)
    listscans, listcts = iohdf5.get_scans_cts(pathHDF5)
    print("number scans found :",len(listscans))
    print("number saved count commands found: ", len(listcts))
    
    dfallscans = None
    pd.set_option('display.max_rows', None)
    dfallscans = pd.DataFrame(listscans, columns=['start_time', 'end_time','sample_dataset_scanindex', 'fullcommand',
                                              'scanindex','scantype','motors','localhdf5file','imagefolder'])
    
    if dfallscans is not None:
        GT.printgreen('Reading of hdf5 file completed\n')
    
    imagefolder = subfolderscandata
    print('For images in imagefolder',imagefolder)
    
    import LaueTools.IOimagefile as IOimage
    blisscommand, imagedate, dallscans_idx = IOimage.fromscmosdate2blisscommand(os.path.join(maindatafolder, imagefolder,'img_0400.tif'),dfallscans)
    print('bliss command was:')
    print(blisscommand)

### set `mapdims` from bliss command 

In [ ]:
# #d5 = iohdf5.build_dict_scan(5, dfallscans, CCDLabel='sCMOS')
# d5 = iohdf5.build_dict_scan_from_imagefile(os.path.join(maindatafolder, imagefolder,'img_0400.tif'), dfallscans, CCDLabel='sCMOS')
# mapdims = d5['mapdimensions']
# invmapdims = mapdims[1], mapdims[0]
# d5

In [ ]:
#!cat {os.path.join(corfilefolder,'img_%04d.cor'%imageindex)}

In [ ]:
#!ls {fitfilefolder}

In [ ]:
#!cat {str(corfilefolder)+'/img_0547_g0.fit'}

# [OPTIONAL] SET new Material (Crystallographic unit cell) for indexing

set `mymaterials`= None if you want to use LaueTools default dict of materials

In [ ]:
mymaterials = None  # use default LaueTools Dict

key_material = 'ZrO2_1200C'

if key_material not in DictLT.dict_Materials.keys():
    msg = f'{key_material} is unknown!\nPlease add your own materials as a dict as in the following:\n '
    msg += '  mymat={"mykey":["mykey", [2,2,2,90,90,120],"no"]}\nwhich should contain '
    msg += '"mykey", a list of lattice parameters and a key for the extinction rules. \n'
    msg += 'See dict_LaueTools and CrystalParameters modules'
    GT.printred(msg)
    
searchedstring = 'Zr'
print(f'\n-------List of key_material containing "{searchedstring}" in the default dictionary-------------------')
print("\n".join([f"'{key}': {DictLT.dict_Materials[key]}," for key in DictLT.dict_Materials.keys() if searchedstring in key]))


#print(f'\n-------List of extinction rules -------------------')
#print("\n".join([str(it) for it in DictLT.dict_Extinc_inv.items()]))


# mymaterials = {'CuZnAl_betabrass': ['CuZnAl_betabrass', [4.4, 5.3, 3.8, 90, 88, 90], 'no'],
# 'CuZnAl_betabrass_light': ['CuZnAl_betabrass_light', [4.4, 5.3, 3.8, 90, 88, 90], 'VO2_mono'],
# 'CuZnAl_tetra': ['CuZnAl_tetra', [4.1, 4.1, 5.8, 90, 90, 90], 'no'],
# 'CuZnAl_6M1': ['CuZnAl_6M1', [4.5, 5.45, 1.3, 94.2, 90, 90], 'no'],
# 'Zn': ['Zn', [2.6649, 2.6649, 4.9468, 90, 90, 120], 'wurtzite'],
# 'ZnO': ['ZnO', [3.252, 3.252, 5.213, 90, 90, 120], 'wurtzite'],
# 'ZnCuOCl': ['ZnCuOCl', [6.839, 6.839, 14.08, 90.0, 90, 120.0], 'SG166'],
# 'ZnCuOCl_all': ['ZnCuOCl_all', [6.839, 6.839, 14.08, 90.0, 90, 120.0], 'no'],}

## [OPTION] set your own materials

In [ ]:
if 0:
    # you may need to prepare your own materials dictionary containing only 'key_material' if 'key_material' is not in the default dictionary
    mymaterials = {'CuZnAl_betabrass': ['CuZnAl_betabrass', [4.4, 5.3, 3.8, 90, 88, 90], 'no'],
                  'ZnO': ['ZnO', [3.252, 3.252, 5.213, 90, 90, 120], 'wurtzite'],
                  'Zn': ['Zn', [2.6649, 2.6649, 4.9468, 90, 90, 120], 'wurtzite']}

    # OR in a parametric way
    c0 = 4.85
    mymaterials = {}
    for kk in range(10):
        mymaterials['Zn%d'%kk]= ['Zn%d'%kk, [2.6649, 2.6649, c0+0.03*kk, 90, 90, 120], 'wurtzite']

    


In [ ]:
if mymaterials is not None:
    print('\n-------- mymaterials  dict --------------')
    print(mymaterials)
else:
    print('`mymaterials` dict is None. So using default dict of materials which should contain your "key_material" label')

## [OPTIONAL] SET some matrices coming from previous or current analyses

In [ ]:
# Guessed UB matrix (example)
UB_1 = np.array([[ 0.800890339,  0.111020812,  0.588618058],
       [ 0.006589284, -0.983840123,  0.176252946],
       [ 0.59871938 , -0.137579654, -0.788529267]])

UB_2 = np.array([[-0.944886962989832, -0.16481648329929 ,  0.282364299451425],
       [ 0.325288903402363, -0.401682702128187,  0.855560084566674],
       [-0.027898445634874,  0.899571897959609,  0.434481129960923]])

UB_3 = np.array([[ 0.798694755,  0.111538191,  0.591397078],
       [ 0.005701942, -0.983734667,  0.177867478],
       [ 0.601654241, -0.138824412, -0.786302534]])

UB_4 = np.array([[-0.944887005, -0.164816017,  0.282364339],
       [ 0.325288815, -0.401682583,  0.855559726],
       [-0.027898058,  0.899571114,  0.434480798]])

UB_5 = np.array([[ 0.790931358,  0.04942662 ,  0.610343974],
       [ 0.006278407, -0.99719131 ,  0.072531087],
       [ 0.611965387, -0.053681208, -0.788581226]])

UB_6 = np.array([[ 0.790951967,  0.049463864,  0.610128265],
       [ 0.006315838, -0.997021885,  0.072650861],
       [ 0.611938365, -0.053678736, -0.788500541]])

# [OPTIONAL] test of refinement only from a proposed UB matrix on a dataset from 1 image

In [ ]:
imageindex = 547

key_material ='Zn'
e_min = 5
e_max = 27 # for refinement
emax_MR = 18 # for indexing step only
previousResults = (2, [UB_2, UB_1], 0, 0)

# Index refine parameters
dict_indexrefine = {# spots set A:  [0,2,58,34] CAUTION max(A) < NBMAXPROBED
                    'central spots indices': [0,1,2],
                    'AngleTolLUT': 0.1,   # Tolerance angle [deg] to recognise angles in the Angular LUT Database
                    'nlutmax': 4,         # Maximum miller index in the Angular LUT Database
                    #number of most intense spot candidate to have a recognisable distance
                    'NBMAXPROBED': 5, # spots set B alias of [0, ..., NBMAXPROBED-1]
                    'MATCHINGRATE_ANGLE_TOL': 0.3,
                    'MinimumMatchingRate': 0,  # minimum MR to accept a previousresults matrix for further refinement
                    'MinimumNumberMatches': 20,  # minimum absolute number of matches (laue spots) keep in list of potential UB
                    'CheckOrientation': None, # os.path.join(working_dir,'UBmat.ubs'),
                    'MATCHINGRATE_THRESHOLD_IAL': 2,  #
                    'GuessedUBMatrix': None,
                    # During Refinement steps: Angular tolerance list for auto links between exp. - theo. spots
                    'list matching tol angles': [0.5,0.2,0.1],#,0.1], 
                    'UseIntensityWeights': False,
                    'nbSpotsToIndex': 10000, # during refinement max number of pairs exp.-theo. spots
                    }

# list of minimum matching rates that accept to perform the next refinement step (with less tolerance angle)
# Last term is the final minimum matching rate (in percent) to accept the final refinement.
MatchingRate_List = [2]*len(dict_indexrefine['list matching tol angles'])
# len(MatchingRate_List) > len('list matching tol angles')

# ---------------end of user parameters----------------------------------------
pathfilecor = os.path.join(corfilefolder,'img_%04d.cor'%imageindex)



t0 = time.time()
# (Re)Initialize object spotsset
dataset = ISS.spotsset()

# Init the dataset with laue spots of .cor file
info = dataset.importdatafromfile(pathfilecor, verbose=0)
print('import dataset from .cor file',info)

# missing info from the dictionary filled manually
dataset.key_material = key_material
dataset.emin = e_min
dataset.emax = e_max
dataset.emax_MR = emax_MR

# index the current dataset
dataset.IndexSpotsSet(None,key_material,e_min,dataset.emax,dict_indexrefine,None,
                              use_file=0, # if 1 , reinit dataset and first argument must be a path to .cor file
                              IMM=False,
                              n_LUT = dict_indexrefine['nlutmax'],
                              LUT = None,
                              angletol_list = dict_indexrefine['list matching tol angles'],
                              nbGrainstoFind = 1, 
                              previousResults = previousResults,
                              dirnameout_fitfile = fitfilefolder,
                              corfilename = pathfilecor,
                              verbose = 0,
                              MatchingRate_List = MatchingRate_List,
                             dictmaterials = mymaterials,
                     choose_UB_MinEulerepresentative=False)

#dataset.dict_grain_devstrain, dataset.dict_grain_latticeparameters, dataset.key_material, dataset.dict_grain_matching_rate

print(f'---- Results of indexing image #{imageindex} ----')
if previousResults:
    print('Starting first by checking previousResults: \n', previousResults, '\n')
print("-------- initial nb of spots",dataset.nbspots)
print("dict of UB matrix found",dataset.dict_grain_matrix)
print('dict of nb of indexed spots and matching rate (%):',dataset.dict_grain_matching_rate)

print(f'For energy max {dataset.emax} keV, dict of nb of indexed spots and matching rate (%):',dataset.dict_grain_matching_rate)
pixelresidues = dataset.pixelresidues
if pixelresidues is not None:
    meanpixdev = np.mean(pixelresidues)
    GT.printgreen(f'For energy max {dataset.emax} keV, nb of spots for refinements : {len(pixelresidues)}')
    GT.printgreen(f'Mean pixel residue {meanpixdev:.2f} pixel') 
if dataset.dict_grain_matrix is {} or any([v is None for v in dataset.dict_grain_matrix.values()]):
    GT.printred('... Nothing found ...')
print(f'\nElapsed time {time.time()-t0:.2f} sec  with internal multiprocessing of {nbmaxcpus} cpus')

# INDEX & REFINE (Multiprocessing) of multiple .cor files

**write as many img_####_g0.fit from img_####.cor file if strain refinement quality is satisfactory**

## SET elementary function (single cpu computation) with user-defined parameters

In [ ]:
# General parameters for indexing
# Exhaustive list
# these parameters can be changed later on
e_min = 5   # [keV]
e_max = 24  # [keV]  for strain refinement
e_max_MR = 18  # [keV] for Matching rate during indexing step
key_material = 'Cr_ref'

nbGrainstoFind = 1
MinimumNumberMatches = 30 # Zr
FinalToleranceRefinement = 0.2
AngleTolLUT=0.2

depth = 0  # microns (normal to surface sample)
fitfilefolder = None

# Index refine parameters
dict_indexrefine = {# spots set A:  [0,2,58,34] CAUTION max(A) < NBMAXPROBED
                    'central spots indices': [0,1,2,3,4],
                    'AngleTolLUT': AngleTolLUT,   # Tolerance angle [deg] to recognise angles in the Angular LUT Database
                    'nlutmax': 3,         # Maximum miller index in the Angular LUT Database
                    #number of most intense spot candidate to have a recognisable distance
                    'NBMAXPROBED': 15, # spots set B alias of [0, ..., NBMAXPROBED-1]
                    'MATCHINGRATE_ANGLE_TOL': 0.2,  # deg, tolerance angle for the first MR computation to test if it is worth being considered for refinement
                    'MinimumMatchingRate':5,  # minimum MR to accept a previousresults matrix for further refinement
                    'MinimumNumberMatches': MinimumNumberMatches,  # minimum absolute number of matches (laue spots) keep in list of potential UB
                    'CheckOrientation': None, # os.path.join(working_dir,'UBmat.ubs'),
                    'MATCHINGRATE_THRESHOLD_IAL': 2,  #
                    'GuessedUBMatrix': None,
                    # During Refinement steps: Angular (deg) tolerance list for auto links between exp. - theo. spots
                    'list matching tol angles': [AngleTolLUT,.5*(AngleTolLUT+FinalToleranceRefinement),FinalToleranceRefinement],#,0.1], 
                    'UseIntensityWeights': False,
                    'nbSpotsToIndex': 10000, # during refinement max number of pairs exp.-theo. spots
                    }

# list of minimum matching rates that accept to perform the next refinement step (with less tolerance angle)
# Last term is the final minimum matching rate (in percent) to accept the final refinement.
MatchingRate_List = [2]*len(dict_indexrefine['list matching tol angles'])
# len(MatchingRate_List) > len('list matching tol angles')

# TO AVOID LOW INFORMATION DATA   or COMPLICATED DATA
MIN_NUMBERSPOTS_FOR_INDEXING = 6
MAX_NUMBERSPOTS_FOR_INDEXING = 10000

# max number of spots to read in  .cor file (None)
MAXNBSPOTS = None  # None

usepreviousUB=False

starting_grainindex = 0
dirnameout_fitfile = fitfilefolder

crudeMReval = True
maxnbspots_MReval = 10000
printindexingstats=False # (interesting for useinternalmultiprocessing = False)
stop_Nb_Matches = 60

useinternalmultiprocessing=False
verbosefilename = True



# building LUT
if mymaterials is not None and key_material in mymaterials:
    latticeparameters = mymaterials[key_material][1]
else:
    latticeparameters = DictLT.dict_Materials[key_material][1]
LUT = IAL.build_AnglesLUT_fromlatticeparameters(latticeparameters, dict_indexrefine['nlutmax'])

#-----------------------------------------------------------------------------
# Elementary function to iterate over for serialization
def index_refine(filename, ignorefitfileresults=True, usepreviousUB=False,
                 skipindexing=True, useinternalmultiprocessing=True, writefitfile=True,
                 LUT= None,
                 verboselevel=0):
    """
    index and refine Laue spots position a .cor file.
    
    According to `ignorefitfile`, the analysis of the .cor file can be omitted
                if previous results written in a corresponding .fit file already exists 
    
    Indexing from scratch step can be skipped by inputing orientation matrix(ces) from previous results
    
    
    ignorefitfileresults: bool, 
        True, the dataset with be analysed (or reananalysed) whatever the existence of previous results
                in corresponding .fit files (existing fit files will be updated and overwritten).
        False, consider only .cor file without any previous results written in the corresponding .fit file.
                Dataset in .cor with already existing results in .fit file won't be (re)analysed
                    T
    usepreviousUB: 'fromfitfile' UB matrix from previous results written in corresponding .fit file will be used
                            for refinement for the currrent dataset in .cor file. If refinment is not successful,
                            then indexing from scratch is started.
                    list of 3x3 matrice, _ ,_) prev to test UB matrix or matrices for successful refinement
                    before finally indexing from scratch if not successful.
                    False, None: indexing from scratch directly
                    
    skipindexing: bool, after poor refinement of previous UB results, indexing from scratch is inhibited. This a check and refine orientation mode
    
    useinternalmultiprocessing: True when calling this function and acceleration with mpi, False when calling it and process wiht 1 cpu

    writefitfile: bool, write results .fit file, if refinement results are satisfactory
    
    global variables defined above:
    filecor_temp
    fitfilefolder
    dict_indexrefine  with its parameters
    previousResults
    key_material
    e_min
    e_max
    nbGrainstoFind
    verboselevel
    MIN_NUMBERSPOTS_FOR_INDEXING
    MAX_NUMBERSPOTS_FOR_INDEXING
    MatchingRate_List
    """
    if isinstance(filename, str):
        filename = Path(filename)
    
    filecor_temp = filename.name
    try:
        image_index = GT.getfileindex(filecor_temp)
    except AttributeError:
        image_index = None

    if not Path(filename).exists():
        return [image_index, [],[], 'missing .cor file']
                    
    dataset_temp = ISS.spotsset()
    dataset_temp.useinternalmultiprocessing = useinternalmultiprocessing
    dataset_temp.stop_Nb_Matches = stop_Nb_Matches
    
    info = dataset_temp.importdatafromfile(filename, verbose=verboselevel,maxnbspots=MAXNBSPOTS)
    #print('info: ', info)
    
    if 'empty' in info:
        return [0, [],[], '.cor file is empty!']
    
    dataset_temp.key_material = key_material
    dataset_temp.emin = e_min
    dataset_temp.emax = e_max
    dataset_temp.emax_MR = e_max_MR
    dataset_temp.crudeMReval = crudeMReval
    dataset_temp.maxnbspots_MReval = maxnbspots_MReval
    dataset_temp.printindexingstats = printindexingstats
    
    if dataset_temp.nbspots < MIN_NUMBERSPOTS_FOR_INDEXING and dataset_temp.nbspots > MAX_NUMBERSPOTS_FOR_INDEXING:
        return [dataset_temp.nbspots, [],[], 'to few or too many spots to index']
    
    previousResults = None
    relatedfitfile = Path(fitfilefolder)/(filecor_temp[:-4]+'_g0.fit')
    info =''
    if not ignorefitfileresults:
        if relatedfitfile.exists():
            return [image_index, [],[], 'not reanalyzed']
        else:
            info='.fit file was missing. '
                
                
    if usepreviousUB == 'fromfitfile':
        if relatedfitfile.exists():
            # read matrix from fit file
            # set previousResults so to skip indexing and go directly to refinement step
            
            #print('read matrix and etc... to be implemented %s'%relatedfitfile)
            UB_g0 = IOLT.readfitfile_multigrains(relatedfitfile)[3].reshape((3,3))
            #print(UB_g0)
            previousResults = (1, [np.array(UB_g0)], 0, 0)

            info = 'Tried UB matrix from .fit file. '

        else:
            info = 'Could not try UB matrix because .fit file was missing. '
    elif usepreviousUB in (False, None):
        pass
    elif isinstance(usepreviousUB, (list, tuple)):
        previousResults = (len(usepreviousUB), usepreviousUB, 0, 0)
    else:
        raise ValueError("%s should be in [False, 4 elemts tuple, 'fromfitfile']"%usepreviousUB)
                

    if skipindexing and usepreviousUB:
        dataset_temp.inhibitindexing = True
        info += ' Check orientation and refine only (no indexing from scratch).'

    Pdirnameout_fitfile = Path(dirnameout_fitfile)
    Pdirnameout_fitfile.mkdir(parents=True, exist_ok=True)

    #print('dataset_temp.crudeMReval',dataset_temp.crudeMReval)

    dataset_temp.IndexSpotsSet(None,
                               key_material,
                               e_min,
                               dataset_temp.emax,
                               dict_indexrefine,
                               None,
                               starting_grainindex=starting_grainindex,
                               use_file=0,
                               IMM = False,
                               n_LUT = dict_indexrefine['nlutmax'],
                               LUT = LUT,
                               angletol_list = dict_indexrefine['list matching tol angles'],
                               nbGrainstoFind = nbGrainstoFind, 
                               previousResults = previousResults,
                               dirnameout_fitfile = Pdirnameout_fitfile,
                               corfilename = filecor_temp,
                               verbose = verboselevel,
                               MatchingRate_List = MatchingRate_List,
                               depth=depth,
                              dictmaterials = mymaterials,
                              writefitfile=writefitfile)
                      
    info+=' Treated by IndexSpotsSet().'

    if 1:
        print(f'grain #{starting_grainindex+0}, #spotindex',
              dataset_temp.getSpotsFamily(starting_grainindex+0))
    
    return [image_index,
        [(key, value) for key, value in (dataset_temp.dict_grain_matrix).items()],  # UB matrix
        [(key, value) for key, value in (dataset_temp.dict_grain_matching_rate).items()], # Nb spots indexed and Matching Rate
       info] 

def index_refine2(filename):
    """
    index and refine Laue spots position a .cor file.
    
    
    
    global variables defined above:
    filecor_temp
    fitfilefolder
    dict_indexrefine  with its parameters
    previousResults
    key_material
    e_min
    e_max
    nbGrainstoFind
    verboselevel
    MIN_NUMBERSPOTS_FOR_INDEXING
    MAX_NUMBERSPOTS_FOR_INDEXING
    MatchingRate_List

    ...
    """    
    if isinstance(filename, str):
        filename = Path(filename)
    
    filecor_temp = filename.name
    image_index = GT.getfileindex(filecor_temp)

    if verbosefilename:
        print(f'Considering {filecor_temp}')

    if not Path(filename).exists():
        return [image_index, [],[], 'missing .cor file']
                    
    dataset_temp = ISS.spotsset()
    dataset_temp.useinternalmultiprocessing = useinternalmultiprocessing
    dataset_temp.stop_Nb_Matches = stop_Nb_Matches
    
    info = dataset_temp.importdatafromfile(filename, verbose=verboselevel,maxnbspots=MAXNBSPOTS)
    #print('info: ', info)
    
    if 'empty' in info:
        return [0, [],[], '.cor file is empty!']
    
    dataset_temp.key_material = key_material
    dataset_temp.emin = e_min
    dataset_temp.emax = e_max
    dataset_temp.emax_MR = e_max_MR
    dataset_temp.crudeMReval = crudeMReval
    dataset_temp.maxnbspots_MReval = maxnbspots_MReval
    dataset_temp.printindexingstats = printindexingstats
    
    if dataset_temp.nbspots < MIN_NUMBERSPOTS_FOR_INDEXING and dataset_temp.nbspots > MAX_NUMBERSPOTS_FOR_INDEXING:
        return [dataset_temp.nbspots, [],[], 'to few or too many spots to index']
    
    previousResults = None
    relatedfitfile = Path(fitfilefolder)/(filecor_temp[:-4]+'_g0.fit')
    info =''
    if not ignorefitfileresults:
        if relatedfitfile.exists():
            return [image_index, [],[], 'not reanalyzed']
        else:
            info='.fit file was missing. '
                
                
    if usepreviousUB == 'fromfitfile':
        if relatedfitfile.exists():
            # read matrix from fit file
            # set previousResults so to skip indexing and go directly to refinement step
            
            #print('read matrix and etc... to be implemented %s'%relatedfitfile)
            UB_g0 = IOLT.readfitfile_multigrains(relatedfitfile)[3].reshape((3,3))
            #print(UB_g0)
            previousResults = (1, [np.array(UB_g0)], 0, 0)

            info = 'Tried UB matrix from .fit file. '

        else:
            info = 'Could not try UB matrix because .fit file was missing. '
    elif usepreviousUB in (False, None):
        pass
    elif isinstance(usepreviousUB, (list, tuple)):
        previousResults = (len(usepreviousUB), usepreviousUB, 0, 0)
    else:
        raise ValueError("%s should be in [False, 4 elemts tuple, 'fromfitfile']"%usepreviousUB)
                

    if skipindexing and usepreviousUB:
        dataset_temp.inhibitindexing = True
        info += ' Check orientation and refine only (no indexing from scratch).'

    Pdirnameout_fitfile = Path(dirnameout_fitfile)
    Pdirnameout_fitfile.mkdir(parents=True, exist_ok=True)

    #print('dataset_temp.crudeMReval',dataset_temp.crudeMReval)

    dataset_temp.IndexSpotsSet(None,
                               key_material,
                               e_min,
                               dataset_temp.emax,
                               dict_indexrefine,
                               None,
                               starting_grainindex=starting_grainindex,
                               use_file=0,
                               IMM = False,
                               n_LUT = dict_indexrefine['nlutmax'],
                               LUT = LUT,
                               angletol_list = dict_indexrefine['list matching tol angles'],
                               nbGrainstoFind = nbGrainstoFind, 
                               previousResults = previousResults,
                               dirnameout_fitfile = Pdirnameout_fitfile,
                               corfilename = filecor_temp,
                               verbose = verboselevel,
                               MatchingRate_List = MatchingRate_List,
                               depth=depth,
                              dictmaterials = mymaterials,
                              writefitfile=writefitfile)

    print(f'grain #{starting_grainindex+0}, #spotindex',dataset_temp.getSpotsFamily(starting_grainindex+0))
    print(f'grain #{starting_grainindex+1}, #spotindex',dataset_temp.getSpotsFamily(starting_grainindex+1))
                      
    info+=' Treated by IndexSpotsSet().'
    
    return [image_index,
        [(key, value) for key, value in (dataset_temp.dict_grain_matrix).items()],  # UB matrix
        [(key, value) for key, value in (dataset_temp.dict_grain_matching_rate).items()], # Nb spots indexed and Matching Rate
       info] 

### [LOOK] at some dataset in a .cor file

In [ ]:
# Plot laue pattern of a given image
if 0:
    print('corfilefolder is', corfilefolder)
    imageindex = 3000
    
    filename = os.path.join(corfilefolder,'img_%04d.cor'%imageindex)
    from LaueTools.LaueDataResultsPlots import PlotPeakPos
    ff, aa = PlotPeakPos(filename, frame='pixel', showindex=False,)#True)
    if aa: aa.grid()

In [ ]:
UB_IMG208_Zr_G0 = np.array([[-0.112798277273925,  0.264283768936951, -0.959088148240022],
       [-0.877156117029428,  0.42783166935261 ,  0.220801693524999],
       [ 0.466630682114979,  0.865400738208747,  0.184125402405369]])

UB_IMG208_Zr_G1 = np.array([[-0.562705662991257, -0.506301694554734,  0.665247385476388],
       [ 0.415142984485895,  0.528172588649008,  0.743784341650145],
       [-0.719449839503281,  0.67927978351221 , -0.090500747250987]])
UB_IMG_205_G1 = np.array([[-0.516363540883251,  0.307665657576283, -0.797216756790789],
       [-0.601836295002085,  0.527854425116734,  0.599381003752638],
       [ 0.609026913819394,  0.793376636209741, -0.087308473127964]])
UB_259 = np.array([[-0.557867151443495, -0.795750512160686,  0.244442422530371],
       [-0.520505141599872,  0.562076465369887,  0.641499591630919],
       [-0.646978346331528,  0.231597087677089, -0.725767034683252]])
UB_269 = np.array([[-0.112703133160243,  0.264388221745783, -0.959196233058513],
       [-0.877117569928531,  0.427892276014889,  0.22076095328233 ],
       [ 0.466726119675911,  0.865254495936136,  0.184119910254875]])

UB_275 = np.array([[-0.113180544631654,  0.263473707596578, -0.959258848306088],
       [-0.877214237188706,  0.427278082255405,  0.221286229492941],
       [ 0.466428820807118,  0.865351376268245,  0.184347402849626]])

UB_277 = np.array([[-0.796892278514895, -0.418385992456327, -0.444032005207359],
       [ 0.125405032242812, -0.822248179289292,  0.554246490939731],
       [-0.590966116554159,  0.379689719474418,  0.705497427650818]])

UB_276 = np.array([[-0.099087799344665,  0.866739015441018, -0.492689495609321],
       [-0.964786607680811,  0.037622194858899,  0.261014947015799],
       [ 0.243513004295334,  0.498923595942045,  0.831496960311785]])  # 96 peaks!
UB_1000_Zr = np. array([[ 0.286275307169192, -0.661462083579904, -0.69433159892593 ],
       [ 0.956787750565669,  0.234829233010284,  0.170955456963756],
       [ 0.052023340886169, -0.711958980345783,  0.699400588866382]])

## Indexing single file with index_refine

In [ ]:
mymaterials  = None

# True (Default), if corfilefolder and fitfilefolder already defined above
# False, to test on a specific .cor file
CORFILE_LOCATION_ALREADY_DEFINED = False

#!ls {corfilefolder}

In [ ]:
imageindex = 3000 #  276 Zr contains two grains ! UB_275 and UB_277
# # only for test with imageindex=600
# testMAtrix = np.array([[0.8824300572185522, 0.19919854054607591, 0.4243793753282497],
#                        [-0.34470667875219557, 0.8905450642933267, 0.2940815804229471],
#                        [-0.32033857637578, -0.4051652260710348, 0.8546980892790703]])

# testMAtrix = np.array([[-0.226608447299619,  0.570374609888871, -0.790584301952095],
#        [-0.962474093376678, -0.007664865523158,  0.271543986838104],
#        [ 0.149199229971046,  0.822416182466247,  0.550421005463097]])

t0 = time.time()

if not CORFILE_LOCATION_ALREADY_DEFINED:
    #/data/visitor/me1701/bm32/20241113/RAW_DATA/A45ZTAA/A45ZTAA_A45ZTAAwire1_daxms/scan0001/dat_img_0005_4900peaks.cor 
    corfilefolder = '/data/visitor/me1701/bm32/20241113/RAW_DATA/A45ZTAA/A45ZTAA_A45ZTAAwire1_daxms/scan0001'
    fitfilefolder =  corfilefolder
    filename = os.path.join(corfilefolder,'dat_img_0005_4900peaks.cor')
    filename = os.path.join(corfilefolder,'shortdat_img_0005_4900peaks.cor')
else: # default (corfilefolder and fitfilefolder are known)
    filename = os.path.join(corfilefolder,'img_%04d.cor'%imageindex)

ignorefitfileresults = True # redo analysis whatever .fit file results already exist
# ignorefitfileresults = False # perform analysis only if corresonding .Fit file results are missing

writefitfile = True
starting_grainindex = 600

MODE =0 

LUT=None
useinternalmultiprocessing = True
crudeMReval = True

key_material = 'Al2O3_colombo'
#key_material = 'Ge'
dirnameout_fitfile = Path(str(fitfilefolder)+f'_JSM_{key_material}')
dict_indexrefine['central spots indices']=[0] #np.arange().tolist()
dict_indexrefine['NBMAXPROBED']=10
#dict_indexrefine['nlutmax']=4
#FinalToleranceRefinement=0.2
#dict_indexrefine['MinimumNumberMatches']=15
#e_max_MR = 20
#e_max =24

# key_material = 'Cr_ref'
# dirnameout_fitfile = Path(str(fitfilefolder)+f'_JSM_{key_material}')
# dict_indexrefine['central spots indices']=[0,1,2,3,4,5]
# dict_indexrefine['NBMAXPROBED']=20
# dict_indexrefine['nlutmax']=4

# key_material = 'Cr'
# dirnameout_fitfile = Path(str(fitfilefolder)+'_Cr')
# dict_indexrefine['central spots indices']=np.arange(20).tolist()
# dict_indexrefine['NBMAXPROBED']=60  # large
# dict_indexrefine['nlutmax']=3   # 3 !!
verboselevel = 0

#### FROM SCRATCH #############
if MODE == 0:
    nbGrainstoFind = 1  # from 1  to  n 
    usepreviousUB = False
    skipindexing = False

    verboselevel = 1
    
    # some tests to reduce computing time!
    starting_grainindex = 300
    crudeMReval=True 
    printindexingstats=False # (True, interesting for useinternalmultiprocessing = False)
    stop_Nb_Matches= 9000 #
    
###  USE PREVIOUS MATRIX in fit file to ONLY refine  ######
elif MODE == 1:
    nbGrainstoFind = 1
    usepreviousUB = 'fromfitfile'  # will read ####_g0.fit
    skipindexing = True

###  USE PREVIOUS MATRIX from a list of UB matrices FIRST and THEN to ONLY refine ######
elif MODE == 2:
    usepreviousUB = [testMAtrix] #[UB_275, UB_277] #[UB_IMG208_Zr_G0]  # Zr
    skipindexing = True
    nbGrainstoFind = 1
    nbGrainstoFind = max(len(previousResults),nbGrainstoFind)
    verboselevel = 1
    crudeMReval = False  # True does not work !!!!

###  USE PREVIOUS MATRIX in .fit file FIRST and THEN if needed perform indexing from scratch  ###
###  indexing form scratch is launched if there are less proposed matrix than `nbGrainstoFind`
elif MODE == 3:
    nbGrainstoFind = 1
    usepreviousUB = 'fromfitfile'
    skipindexing = False
    
###  USE PREVIOUS MATRIX from a list of UB matrices FIRST and THEN if needed perform indexing from scratch  ###
###  indexing form scratch is launched if there are less proposed matrix than `nbGrainstoFind`
elif MODE == 4:
    usepreviousUB = [UB_1000_Zr]#UB_275, UB_277]  # Zr
    #previousResults = [UB_2, UB_1]  # Zn
    #previousResults = [UB_Zr_828] # Zr
    skipindexing = False
    nbGrainstoFind = max(len(previousResults),nbGrainstoFind)

if max(dict_indexrefine['central spots indices'])>=dict_indexrefine['NBMAXPROBED']:
    GT.printyellow('''dict_indexrefine['NBMAXPROBED'] is too low''')
else:
    res = index_refine(filename,
                       ignorefitfileresults=ignorefitfileresults,
                       usepreviousUB=usepreviousUB, #False,#(1,[UB547],0,0),
                       skipindexing=skipindexing,
                       useinternalmultiprocessing=useinternalmultiprocessing,
                       writefitfile=writefitfile,
                       LUT=LUT,
                       verboselevel=verboselevel)
    
    print('Indexing single dataset (but with multiprocessing acceleration)')
    print(f'image index {res[0]}')
    #print(f'Initial number of spots {res[0]}')
    print(f'UB matrix(ces) found: {res[1]}')
    print(f'Matching rate (grain index, [Nb of indexed spots, MR (%)]): {res[2]}')
    # if len(res[1]) <= 1:
    #     GT.printgreen(f'UB matrix(ces) found: {res[1]}')
    #     GT.printgreen(f'Matching rate (grain index, [Nb of indexed spots, MR (%)]): {res[2]}')
    # else:
    #     GT.printred(f'UB matrix(ces) found: {res[1]}')
    #     GT.printred(f'Last Matching rate (grain index, [Nb of indexed spots, MR (%)]): {res[2]}')
    print('\nInfo: ', res[-1])
    print('Elapsed time: %.2f seconds'%(time.time()-t0))

warnings.filterwarnings('default')

In [ ]:
# ME1701:
array([[ 0.865624188536201, -0.012593137682994,  0.499505408709559],
       [-0.437509815497355,  0.468596439236897,  0.765703856363049],
       [-0.244046759311608, -0.884557281754983,  0.398573295888568]]))

In [ ]:
dirnameout_fitfile

In [ ]:
ub= np.array([[-0.20282682176412,  -0.856760678374662, -0.466056031401537],
 [ 0.063135051186231, -0.545796980850474,  0.828792132342705],
 [ 0.02043610463695,   0.813270765959301,  0.56879228020785 ]])

In [ ]:
np.linalg.det(ub*1.75)

In [ ]:
ub*1.75

In [ ]:
ub_good = np.array([[ 0.895128429516913,  0.100534265557968, -0.433323777620105],
       [-0.053293939649094,  0.989123976918054,  0.118070933722633],
       [ 0.438937600518584, -0.083197175740935,  0.893608795411921]])

In [ ]:
np.linalg.det(ub_good)

In [ ]:
ub_good[:,0].dot(ub_good[:,1]), ub_good[:,0].dot(ub_good[:,2]),ub_good[:,1].dot(ub_good[:,2])

In [ ]:
np.linalg.norm(ub_good[:,0]),np.linalg.norm(ub_good[:,1]),np.linalg.norm(ub_good[:,2])

In [ ]:
def is_ubmatrix_distorted_orthogonal_rotation(ubmatrix, tolerance = 0.05):
    """
    Check if ubmatrix is a distorted orthogonal rotation matrix
    (i.e. check if it is close enough to an orthogonal rotation matrix)

    q = ubmatrix B0 G*  where B0 (triangular up matrix) comes from lattice parameters input

    :param ubmatrix: 3x3 matrix to be tested
    :return: boolean, True if ubmatrix satisfies the conditions, False otherwise
    """

    dot_products = np.array([
        np.abs(ubmatrix[:, 0].dot(ubmatrix[:, 1])),
        np.abs(ubmatrix[:, 0].dot(ubmatrix[:, 2])),
        np.abs(ubmatrix[:, 1].dot(ubmatrix[:, 2]))
    ])

    norms = np.array([
        np.abs(ubmatrix[:, 0].dot(ubmatrix[:, 0])),
        np.abs(ubmatrix[:, 1].dot(ubmatrix[:, 1])),
        np.abs(ubmatrix[:, 2].dot(ubmatrix[:, 2]))
    ])

    conditions = (dot_products < tolerance) & (np.abs(norms - 1) < tolerance)

    return np.all(conditions)

In [ ]:
is_ubmatrix_distorted_orthogonal_rotation(ub) == True

In [ ]:
len(res)

In [ ]:
print('fit files are written in :',dirnameout_fitfile)

## Indexing single file with index_refine2

In [ ]:
imageindex = 25 #  276 Zr contains two grains ! UB_275 and UB_277

t0 = time.time()
filename = os.path.join(corfilefolder,'img_%04d.cor'%imageindex)
# if os.stat(filename).st_size<=17:
#     GT.printred('.cor file is empty ...')
#     GT.printred('Do a test on other file please ...\n\n')

ignorefitfileresults = True # redo analysis whatever .fit file results already exist
# ignorefitfileresults = False # perform analysis only if corresonding .Fit file results are missing

writefitfile = True

starting_grainindex = 500

MODE = 0 #0
verboselevel = 0
verbosefilename = True
LUT=None
crudeMReval = True

key_material = 'Zr-ref'
dirnameout_fitfile = Path(str(fitfilefolder)+f'_{key_material}')
dict_indexrefine['central spots indices']=[0,1,2,3,4]
dict_indexrefine['NBMAXPROBED']=20
dict_indexrefine['nlutmax']=3

# key_material = 'Cr'
# dirnameout_fitfile = Path(str(fitfilefolder)+'_Cr')
# dict_indexrefine['central spots indices']=np.arange(20).tolist()
# dict_indexrefine['NBMAXPROBED']=60  # large
# dict_indexrefine['nlutmax']=3   # 3 !!

#### FROM SCRATCH #############
if MODE == 0:
    nbGrainstoFind = 2  # from 1  to  n 
    previousResults = None
    skipindexing = False
    
    # some tests to reduce computing time!
    starting_grainindex = 300
    crudeMReval=False # True does not work
    maxnbspots_MReval, stop_Nb_Matches = 200, 10
    #maxnbspots_MReval, stop_Nb_Matches = 10000, 50
    
    printindexingstats=False # (True, interesting for useinternalmultiprocessing = False)

    
###  USE PREVIOUS MATRIX in fit file to ONLY refine  ######
elif MODE == 1:
    nbGrainstoFind = 1
    previousResults = 'fromfitfile'  # will read ####_g0.fit
    skipindexing = True

###  USE PREVIOUS MATRIX from a list of UB matrices FIRST and THEN to ONLY refine ######
elif MODE == 2:
    usepreviousUB = [testMAtrix]
    skipindexing = True
    nbGrainstoFind = max(len(usepreviousUB),nbGrainstoFind)
    crudeMReval=False # True does not work !!!!!
    verboselevel = 2

###  USE PREVIOUS MATRIX in .fit file FIRST and THEN if needed perform indexing from scratch  ###
###  indexing form scratch is launched if there are less proposed matrix than `nbGrainstoFind`
elif MODE == 3:
    nbGrainstoFind = 1
    previousResults = 'fromfitfile'
    skipindexing = False

###  USE PREVIOUS MATRIX from a list of UB matrices FIRST and THEN if needed perform indexing from scratch  ###
###  indexing form scratch is launched if there are less proposed matrix than `nbGrainstoFind`
elif MODE == 4:
    usepreviousUB = [UB_1000_Zr]#UB_275, UB_277]  # Zr
    #previousResults = [UB_2, UB_1]  # Zn
    #previousResults = [UB_Zr_828] # Zr
    skipindexing = False
    nbGrainstoFind = max(len(previousResults),nbGrainstoFind)

useinternalmultiprocessing=True

print('skipindexing',skipindexing)

res = index_refine2(filename)

print('------- Indexing single dataset  ------')
if useinternalmultiprocessing:
    print('but with multiprocessing ')
else:
    print('without multiprocessing')
print('Stopping level of Nb of Matches', stop_Nb_Matches)
print(f'image index {res[0]}')
#print(f'Initial number of spots {res[0]}')
#print(f'UB matrix(ces) found: {res[1]}')
#print(f'Matching rate (grain index, [Nb of indexed spots, MR (%)]): {res[2]}')

for gub, score in zip(res[1], res[2]):
    gindex, ub = gub
    nir, mr = score[1]
    if nir>20 and mr > 20:
        GT.printgreen(f'Nb indexed Reflections: {nir}, Matching Rate {mr:.2f}%')
    else:
        GT.printred(f'Nb indexed Reflections: {nir}, Matching Rate {mr:.2f}%')
    print('UB matrix found: np.array('+str(ub.tolist())+')\n')
# if len(res[1]) <= 1:
#     GT.printgreen(f'UB matrix(ces) found: {res[1]}')
#     GT.printgreen(f'Matching rate (grain index, [Nb of indexed spots, MR (%)]): {res[2]}')
# else:
#     GT.printred(f'UB matrix(ces) found: {res[1]}')
#     GT.printred(f'Last Matching rate (grain index, [Nb of indexed spots, MR (%)]): {res[2]}')
print('\nInfo: ', res[-1])
print('Elapsed time: %.2f seconds'%(time.time()-t0))

In [ ]:
res

In [ ]:
testMAtrix = np.array([[0.8824300572185522, 0.19919854054607591, 0.4243793753282497], [-0.34470667875219557, 0.8905450642933267, 0.2940815804229471], [-0.32033857637578, -0.4051652260710348, 0.8546980892790703]])

##  test loop with index_refine2

In [ ]:
for imageindex in range(30,64):

    t0 = time.time()
    filename = os.path.join(corfilefolder,'img_%04d.cor'%imageindex)
    # if os.stat(filename).st_size<=17:
    #     GT.printred('.cor file is empty ...')
    #     GT.printred('Do a test on other file please ...\n\n')
    
    ignorefitfileresults = True # redo analysis whatever .fit file results already exist
    # ignorefitfileresults = False # perform analysis only if corresonding .Fit file results are missing
    
    writefitfile = True
    
    starting_grainindex = 500
    
    MODE = 2  #0
    verboselevel = 0
    verbosefilename = True
    LUT=None
    crudeMReval=False   # True does not work !!


    key_material = 'Zr'
    dirnameout_fitfile = Path(str(fitfilefolder)+f'JSM_{key_material}')
    dict_indexrefine['central spots indices']=[0,1,2,3,4]
    dict_indexrefine['NBMAXPROBED']=20
    dict_indexrefine['nlutmax']=3
    
    # key_material = 'Cr'
    # dirnameout_fitfile = Path(str(fitfilefolder)+'_Cr')
    # dict_indexrefine['central spots indices']=np.arange(20).tolist()
    # dict_indexrefine['NBMAXPROBED']=60  # large
    # dict_indexrefine['nlutmax']=3   # 3 !!
    
    #### FROM SCRATCH #############
    if MODE == 0:
        nbGrainstoFind = 2  # from 1  to  n 
        previousResults = None
        skipindexing = False
        
        # some tests to reduce computing time!
        starting_grainindex = 300
        maxnbspots_MReval, stop_Nb_Matches = 200, 10
        #maxnbspots_MReval, stop_Nb_Matches = 10000, 50
        
        printindexingstats=False # (True, interesting for useinternalmultiprocessing = False)
    
        
    ###  USE PREVIOUS MATRIX in fit file to ONLY refine  ######
    elif MODE == 1:
        nbGrainstoFind = 1
        previousResults = 'fromfitfile'  # will read ####_g0.fit
        skipindexing = True
    
    ###  USE PREVIOUS MATRIX from a list of UB matrices FIRST and THEN to ONLY refine ######
    elif MODE == 2:
        previousResults = [nbGrainstoFind]  # from 1  to  n ] #[UB_275, UB_277] #[UB_IMG208_Zr_G0]  # Zr
        skipindexing = True
        nbGrainstoFind = max(len(previousResults),nbGrainstoFind)
    
    ###  USE PREVIOUS MATRIX in .fit file FIRST and THEN if needed perform indexing from scratch  ###
    ###  indexing form scratch is launched if there are less proposed matrix than `nbGrainstoFind`
    elif MODE == 3:
        nbGrainstoFind = 1
        previousResults = 'fromfitfile'
        skipindexing = False
    
    ###  USE PREVIOUS MATRIX from a list of UB matrices FIRST and THEN if needed perform indexing from scratch  ###
    ###  indexing form scratch is launched if there are less proposed matrix than `nbGrainstoFind`
    elif MODE == 4:
        previousResults = [UB_1000_Zr]#UB_275, UB_277]  # Zr
        #previousResults = [UB_2, UB_1]  # Zn
        #previousResults = [UB_Zr_828] # Zr
        skipindexing = False
        nbGrainstoFind = max(len(previousResults),nbGrainstoFind)
    
    useinternalmultiprocessing=True
    
    res = index_refine2(filename)
    
    # print('------- Indexing single dataset  ------')
    if useinternalmultiprocessing:
        print('but with multiprocessing ')
    else:
        print('without multiprocessing')
    # print('Stopping level of Nb of Matches', stop_Nb_Matches)
    # print(f'image index {res[0]}')
    #print(f'Initial number of spots {res[0]}')
    #print(f'UB matrix(ces) found: {res[1]}')
    #print(f'Matching rate (grain index, [Nb of indexed spots, MR (%)]): {res[2]}')
    
    for gub, score in zip(res[1], res[2]):
        gindex, ub = gub
        nir, mr = score[1]
        if nir>20 and mr > 20:
            GT.printgreen(f'Nb indexed Reflections: {nir}, Matching Rate {mr:.2f}%')
        else:
            GT.printred(f'Nb indexed Reflections: {nir}, Matching Rate {mr:.2f}%')
        # print('UB matrix found: np.array('+str(ub.tolist())+')\n')
    # if len(res[1]) <= 1:
    #     GT.printgreen(f'UB matrix(ces) found: {res[1]}')
    #     GT.printgreen(f'Matching rate (grain index, [Nb of indexed spots, MR (%)]): {res[2]}')
    # else:
    #     GT.printred(f'UB matrix(ces) found: {res[1]}')
    #     GT.printred(f'Last Matching rate (grain index, [Nb of indexed spots, MR (%)]): {res[2]}')
    # print('\nInfo: ', res[-1])
    # print('Elapsed time: %.2f seconds'%(time.time()-t0))
    print('res',res)

# MULTIPROCESS INDEX & REFINE  .cor files

In [ ]:
pool

In [ ]:
del(pool)

## MULTIPROCESSING of index_refine2 and useinternalmultiprocessing=False

still slow not using full power of nb of cpus...


In [ ]:
# MULTIPROCESSING of index_refine2 and useinternalmultiprocessing=False

#if __name__=='__main__':
#-------------  user-defined parameters ------------------
nb_cpus =64

# if True reanalyse all .cor images (whatever existing .fit files...) and overwrite results .fit files
# if False, only .cor file with missing corresponding fitfile will be analysed
ignorefitfileresults =True

# if False, indexing from scratch
# if 'fromfitfile', refinement of UB matrix found in corresponding .fit file. If refinement is poor then indexing from scratch
# if elements tuple (nb of matrices, list of UB,0,0)
usepreviousUB = False #(555,[UB_Zr_1],555,555)  # False
#usepreviousUB = (555,[UB_Zr_828],555,555)  # False

# inhibit or not indexing from scratch after having tested previous results
# True if one wants to catch only 1 grain
# False to allow indexing from scratch
skipindexing = False

# generally True to save permanently results on disk with .fit file
writefitfile = True

verbosefilename = False
verboselevel = 0

MODE = 0 #0

#### FROM SCRATCH #############
if MODE == 0:
   
    nbGrainstoFind = 1  # from 1  to  n 
    previousResults = None
    skipindexing = False
    
    # some tests to reduce computing time!
    starting_grainindex = 0

    # # Cr ----------
    # key_material = 'Cr'
    # dirnameout_fitfile = Path(str(fitfilefolder)+'_Cr')
    # dict_indexrefine['central spots indices']=np.arange(4).tolist()
    # dict_indexrefine['NBMAXPROBED']=30  # large
    # dict_indexrefine['nlutmax']=3   # 3 !!
    
    verboselevel = 0
    key_material = 'Zr'
    dirnameout_fitfile = Path(str(fitfilefolder)+'_Zr')
    dict_indexrefine['central spots indices']=[0,1,2,3]
    dict_indexrefine['NBMAXPROBED']=12
    dict_indexrefine['nlutmax']=3

    crudeMReval=False
    printindexingstats=False # (True, interesting for useinternalmultiprocessing = False)
    stop_Nb_Matches =60 #

    # --------------------------
elif MODE == 2:
    
    key_material = 'Zr'
    previousResults = [testMAtrix] #[UB_275, UB_277] #[UB_IMG208_Zr_G0]  # Zr
    # ----------------
    nbGrainstoFind = 1  # from 1  to  n
    skipindexing = True
    nbGrainstoFind = max(len(previousResults),nbGrainstoFind)

    
#-----------set `listfiles` ----------------------------------
#cor_filelist = [os.path.join(corfilefolder,_file) for _file in os.listdir(corfilefolder) if (_file.startswith("img_") and _file.endswith(".cor"))]
if usepreviousUB is None:
    raise ValueError('usepreviousUB can not be None. You meant False')

# -------   define `listfiles` (SELECT set of .cor files)  --------------
# Choice A
# use better Path.glob('*.cor') ...
cor_filelist = [str(ff) for ff in sorted(Path(corfilefolder).glob('img*.cor'))]

# Choice B
# to test or handle only a subset of the list of files by image index
startindex, finalindex = 29300, 30100# + 3*nb_cpus
# a list of indices
listindices = np.arange(startindex,finalindex+1,1)

# a list of indices corresponding to a rectangle in the sample map (ROI)
listindices = GT.extract2Dslice(1070, (20, 20), np.arange(mapdims[0]*mapdims[1]).reshape(invmapdims)).ravel()
selectedlistfiles = [ff for ff in cor_filelist if GT.getfileindex(ff) in listindices]

### final choice of the list of files  user choice of `listindices`
listfiles= cor_filelist[:6000]#  or even listfiles = cor_filelist[547:547+2*nb_cpus]
#listfiles = selectedlistfiles
# ------- END of define `listfiles`  --------------

# ---------------------------------------------------------
#---------END OF user-defined parameters ------------------
#----------------------------------------------------------
# building LUT
if mymaterials is not None and key_material in mymaterials:
    latticeparameters = mymaterials[key_material][1]
else:
    latticeparameters = DictLT.dict_Materials[key_material][1]
LUT = IAL.build_AnglesLUT_fromlatticeparameters(latticeparameters, dict_indexrefine['nlutmax'])

nb_corfiles = len(listfiles)

args_index_refine = zip(listfiles,
               itertools.repeat(ignorefitfileresults),
               itertools.repeat(usepreviousUB),
               itertools.repeat(skipindexing),
                        itertools.repeat(False),
                        itertools.repeat(writefitfile),
                        itertools.repeat(LUT),
                       itertools.repeat(verboselevel))

args_index_refine2 = listfiles

t4 = time.time()

with multiprocessing.Pool(nb_cpus) as pool:
    allresults = pool.starmap(index_refine,
                 tqdm(args_index_refine, total = nb_corfiles, desc = 'Indexation and Refinement progress'),
                 chunksize = 1)
# with multiprocessing.Pool(nb_cpus) as pool:
#     allresults = pool.starmap(index_refine2,
#                  tqdm(args_index_refine2, total = nb_corfiles, desc = 'Indexation and Refinement progress'),
#                  chunksize = 1)

# with multiprocessing.Pool(processes=nb_cpus) as pool:
#         allresults = []
#         # imap_unordered yields results as soon as they finish
#         for r in tqdm(pool.imap_unordered(index_refine2, listfiles, chunksize=1), total=nb_corfiles):
#             allresults.append(r)

t5 = time.time()
minutes = int(np.ceil(t5 - t4) // 60)
seconds = int(np.ceil(t5 - t4)  % 60)

print(f"It took {minutes}mins and {seconds}s to index {nb_corfiles} images with {nb_cpus} cpus.")
if not multiprocessing.active_children():
    GT.printgreen("Done!\n\n")

## Loop on index_refine2 with internal mutliprocessing

see getOrientMatrices() in IndexingAnglesLut.py

In [ ]:
# loop of index_refine2 and useinternalmultiprocessing=False   9 sec /image (2 grains of Zr)
# loop of index_refine2 and useinternalmultiprocessing=True   9 sec /image (2 grains of Zr)

#-------------  user-defined parameters ------------------
nb_cpus = 64

# if True reanalyse all .cor images (whatever existing .fit files...) and overwrite results .fit files
# if False, only .cor file with missing corresponding fitfile will be analysed
ignorefitfileresults =True

# if False, indexing from scratch
# if 'fromfitfile', refinement of UB matrix found in corresponding .fit file. If refinement is poor then indexing from scratch
# if elements tuple (nb of matrices, list of UB,0,0)
usepreviousUB = False #(555,[UB_Zr_1],555,555)  # False
#usepreviousUB = (555,[UB_Zr_828],555,555)  # False

# inhibit or not indexing from scratch after having tested previous results
# True if one wants to catch only 1 grain
# False to allow indexing from scratch
skipindexing = False

# generally True to save permanently results on disk with .fit file
writefitfile = True

MODE = 0

#### FROM SCRATCH #############
if MODE == 0:
   
    nbGrainstoFind = 2  # from 1  to  n 
    previousResults = None
    skipindexing = False
    
    # some tests to reduce computing time!
    starting_grainindex = 0

    # # Cr ----------
    # key_material = 'Cr'
    # dirnameout_fitfile = Path(str(fitfilefolder)+'_Cr')
    # dict_indexrefine['central spots indices']=np.arange(4).tolist()
    # dict_indexrefine['NBMAXPROBED']=30  # large
    # dict_indexrefine['nlutmax']=3   # 3 !!
    
    verboselevel = 0
    key_material = 'Zr'
    dict_indexrefine['nlutmax']=3
    dirnameout_fitfile = Path(str(fitfilefolder)+'_Zr')
    if 0:  # 1 image / 15 s
        dict_indexrefine['central spots indices']=[0,1,2,3,4,5]
        dict_indexrefine['NBMAXPROBED']=20
    if 0:  # 1 image / 12 s but less results
        dict_indexrefine['central spots indices']=[0,1,2]
        dict_indexrefine['NBMAXPROBED']=20
    if 1:  # 1
        dict_indexrefine['central spots indices']=[0,1]
        dict_indexrefine['NBMAXPROBED']=40
    

    crudeMReval=True
    printindexingstats=False # (True, interesting for useinternalmultiprocessing = False)
    #stop_Nb_Matches = 60
    maxnbspots_MReval, stop_Nb_Matches = 250, 30

    # --------------------------

#-----------set `listfiles` ----------------------------------
#cor_filelist = [os.path.join(corfilefolder,_file) for _file in os.listdir(corfilefolder) if (_file.startswith("img_") and _file.endswith(".cor"))]
if usepreviousUB is None:
    raise ValueError('usepreviousUB can not be None. You meant False')

# -------   define `listfiles` (SELECT set of .cor files)  --------------
# Choice A
# use better Path.glob('*.cor') ...
cor_filelist = [ff for ff in sorted(corfilefolder.glob('img*.cor'))]

# Choice B
# to test or handle only a subset of the list of files by image index
startindex, finalindex = 29300, 30100# + 3*nb_cpus
# a list of indices
listindices = np.arange(startindex,finalindex+1,1)

# a list of indices corresponding to a rectangle in the sample map (ROI)
listindices = GT.extract2Dslice(1070, (20, 20), np.arange(mapdims[0]*mapdims[1]).reshape(invmapdims)).ravel()
selectedlistfiles = [ff for ff in cor_filelist if GT.getfileindex(ff) in listindices]

### final choice of the list of files  user choice of `listindices`
listfiles= cor_filelist[:2000]#  or even listfiles = cor_filelist[547:547+2*nb_cpus]
#listfiles = selectedlistfiles
# ------- END of define `listfiles`  --------------

# ---------------------------------------------------------
#---------END OF user-defined parameters ------------------
#----------------------------------------------------------
# building LUT
if mymaterials is not None and key_material in mymaterials:
    latticeparameters = mymaterials[key_material][1]
else:
    latticeparameters = DictLT.dict_Materials[key_material][1]
LUT = IAL.build_AnglesLUT_fromlatticeparameters(latticeparameters, dict_indexrefine['nlutmax'])

nb_corfiles = len(listfiles)

t4 = time.time()

allresults = []

useinternalmultiprocessing=True
for fname in tqdm(listfiles, total=nb_corfiles):
    r = index_refine2(fname)
    allresults.append(r)

t5 = time.time()
minutes = int(np.ceil(t5 - t4) // 60)
seconds = int(np.ceil(t5 - t4)  % 60)

print(f"It took {minutes}mins and {seconds}s to index {nb_corfiles} images with {nb_cpus} cpus.")


In [ ]:
print('Process and summarize results')
        
# Results Summary:
treatedimages = []
# mapdims   fast, slow
invmapdims = mapdims[1],mapdims[0]   # slow, fast
Nbindexedimages2D = np.empty(invmapdims)  # nb of indexed spots 
MRindexedimages2D = np.empty(invmapdims)  # matching rate
Nbindexedimages2D[:]=np.nan
MRindexedimages2D[:]=np.nan
maskindexed2D = False*np.ones(invmapdims, dtype=bool)
grainlistindices = []
nonindexedimages = []
indexedimages = []   # 
n2 =mapdims[0]  # fast axis

print('n2',n2)
print('Nbindexedimages2D.shape', Nbindexedimages2D.shape)

for elem in tqdm(allresults):

    ix, dictmatrix, dictMR, info = elem
    treatedimages.append(ix)
    #print(dictmatrix)
    if len(dictmatrix)>0:
        maskindexed2D[ix//n2, ix%n2]=True
        grainlistindices.append(ix)
        indexedimages.append(ix)
        nbindexed, MR = dictMR[0][1]
        if isinstance(nbindexed, int):
            Nbindexedimages2D[ix//n2, ix%n2]=nbindexed
        if isinstance(MR, float):
            MRindexedimages2D[ix//n2, ix%n2]=MR
    else:
        nonindexedimages.append(ix)
        maskindexed2D[ix//n2, ix%n2]=False

GT.printgreen('Done !')

In [ ]:
del(pool)

### [SAVE] `allresults`

In [ ]:
import pickle, datetime

SAVE_ALLRESULTS_PICKLE = True

saving_folder = corfilefolder
mystamp = 'Zrmap2D'
saving_filename = 'allresults_%s.pickle'%mystamp


#------------end of user inputs -------------
if SAVE_ALLRESULTS_PICKLE: # SAVE
    fullpathsaving = os.path.join(saving_folder,saving_filename)
    
    WRITEFILE=True
    if Path(fullpathsaving).exists():
        GT.printred('The following File already exists!')
        if input('Do you want to overwrite it?') not in ('y','Y','yes','YES'):
            print('Please change the filename if you really want to save your results')
            WRITEFILE=False
            
if WRITEFILE:    
    dict_scan_data = {'maindatafolder':maindatafolder,
                     'subfolderscandata':subfolderscandata,
                      'imagefolder':imagefolder,
                     'datfilefolder':datfilefolder,
                     'corfilefolder':corfilefolder,
                      'fitfilefolder':fitfilefolder,
                     'prefixfilename':prefixfilename,
                     'suffix':suffix,
                     'CCDLabel':CCDLabel,
                     'sizeofzeropadding':sizeofzeropadding,
                     'detfile':detfile,
                     'mapdims':mapdims,
                     'invmapdims':invmapdims}
    
    dict_index_refine_for_mpi ={'e_min':e_min,
                      'e_max':e_max,
                      'e_max_MR':e_max_MR,
                      'key_material':key_material,
                      'nbGrainstoFind':nbGrainstoFind,
                      'depth':depth,
                      'dict_indexrefine':dict_indexrefine,
                      'MatchingRate_List':MatchingRate_List,
                      'MIN_NUMBERSPOTS_FOR_INDEXING':MIN_NUMBERSPOTS_FOR_INDEXING,
                      'MAX_NUMBERSPOTS_FOR_INDEXING':MAX_NUMBERSPOTS_FOR_INDEXING}
    
    dict_info_collect_mpi = {'nb_cpus':nb_cpus,
                            'ignorefitfileresults':ignorefitfileresults,
                            'usepreviousUB':usepreviousUB,
                            'skipindexing':skipindexing,
                            'writefitfile':writefitfile}
    
    dict_results_summary = {'treatedimages':treatedimages,
                           'nonindexedimages':nonindexedimages,
                           'indexedimages':indexedimages,
                           'maskindexed2D':maskindexed2D,
                            'Nbindexedimages2D':Nbindexedimages2D,
                            'MRindexedimages2D':MRindexedimages2D,
                            'grainlistindices':grainlistindices}
    
    dictresults={}
    dictresults['allresults']=allresults
    dictresults['dict_scan_data'] = dict_scan_data
    dictresults['dict_index_refine_for_mpi'] = dict_index_refine_for_mpi
    dictresults['dict_info_collect_mpi'] = dict_info_collect_mpi
    dictresults['dict_results_summary'] = dict_results_summary
    with open(fullpathsaving, 'wb') as f:
        pickle.dump(dictresults, f)
        GT.printgreen(f'\nLast allresults.pickle file was saved at {datetime.datetime.now()}:')
        GT.printgreen(f'{fullpathsaving}')

### [LOAD] `allresults` from "allresults" .pickle file

In [ ]:
if 0: #LOAD
    fullpathsaving = '/data/projects/zndendrites/ma6034/bm32/20240625/RAW_DATA/'
    fullpathsaving += 'sample1_test/sample1_test_sample5-30m-new2-coarse-large/scan0001/'
    fullpathsaving += 'datfiles_notebook/allresults_G218.pickle'
    
    fullpathsaving ='/data/projects/mapgrainxl/blc15488/bm32/20240601/RAW_DATA/ech15/ech15_map2Dexpo0p1sec/scan0001/img_CORfiles/allresults_G1.pickle'

    if input('\n\n\n?? ARE YOU SURE TO LOAD a previous file and OVERWRITE your current dictresults ??\n\n') in ('y','yes','Y','YES','o','O'):
        with open(fullpathsaving, 'rb') as f:
            dictresults=pickle.load(f)
        allresults = dictresults['allresults']
        dict_scan_data = dictresults['dict_scan_data']
        dict_index_refine_for_mpi= dictresults['dict_index_refine_for_mpi']
        dict_indexrefine = dict_index_refine_for_mpi['dict_indexrefine']
        dict_info_collect_mpi = dictresults['dict_info_collect_mpi']
        dict_results_summary = dictresults['dict_results_summary']
        mapdims = dict_scan_data['mapdims']
        invmapdims = dict_scan_data['invmapdims']
        corfilefolder = dict_scan_data['corfilefolder']
        fitfilefolder = dict_scan_data['fitfilefolder']
        depth = dict_index_refine_for_mpi['depth']
        maskindexed2D = dict_results_summary['maskindexed2D']
        usepreviousUB = dict_info_collect_mpi['usepreviousUB']
        treatedimages = dict_results_summary['treatedimages']
        nonindexedimages = dict_results_summary['nonindexedimages']
        indexedimages = dict_results_summary['indexedimages']
        Nbindexedimages2D = dict_results_summary['Nbindexedimages2D']
        MRindexedimages2D = dict_results_summary['MRindexedimages2D']

## Some visualisations of results (to evaluate quality confidence)

In [ ]:
if usepreviousUB:
    print('-------For usepreviousUB-------\n', usepreviousUB)
    print('and last tolerance angle %.2f deg\n'%dict_indexrefine["list matching tol angles"][-1])
    
print('nb of treated datasets :',len(treatedimages))
print('number of non indexed datasets', len(nonindexedimages))
print('number of indexed (or updated) datasets', len(indexedimages))
#print('indexed (or updated) dataset indices list',grainlistindices)

In [ ]:
print('mapdims (fast axis, slow axis)',mapdims)

# depending on scan   dmesh fastmotor .. ..  slowmotor .. ..
motorfastaxis = 'yech'
motorslowaxis = 'xech'

In [ ]:
fig2,ax2 = plt.subplots()
Nbindexedimages2Dplot = copy.copy(Nbindexedimages2D)
pp2 = ax2.imshow(Nbindexedimages2Dplot, origin='lower', vmin=0)
txt = f'{corfilefolder}\nkey_material={key_material}'
ax2.set_title(txt+'\nNb indexed spots')
plt.colorbar(pp2)
GT.format_getimageindex_imshow.__defaults__=(invmapdims,)
ax2.format_coord = GT.format_getimageindex_imshow
ax2.set_xlabel(motorfastaxis)
ax2.set_ylabel(motorslowaxis)

In [ ]:
# some mask conditions
cond_isnotindexed = np.invert(maskindexed2D)

MinNbthreshold = 20
cond_Nbindexed_poor = Nbindexedimages2Dplot<MinNbthreshold

no_mask = False*np.ones_like(maskindexed2D)

# choice for the plot
condG1 = cond_Nbindexed_poor #no_mask #cond_Nbindexed_poor  cond_isnotindexed
condG1 = cond_isnotindexed
#condG1 = no_mask

# using mask of indexed and refined data
fig2,ax2 = plt.subplots()
Nbindexedimages2Dplot = copy.copy(Nbindexedimages2D)
Nbindexedimages2Dplot = ma.masked_where(condG1, Nbindexedimages2Dplot)

pp2 = ax2.imshow(Nbindexedimages2Dplot, origin='lower')
ax2.set_title(txt+'\nNb indexed spots higher than %d '%MinNbthreshold)
plt.colorbar(pp2)
GT.format_getimageindex_imshow.__defaults__=(invmapdims,)
ax2.format_coord = GT.format_getimageindex_imshow
ax2.set_xlabel(motorfastaxis)
ax2.set_ylabel(motorslowaxis)

In [ ]:
# using mask wrt mininmum number of matching rate (%)
MinMRthreshold = 20 # in %
cond_smallMR = MRindexedimages2D <= MinMRthreshold

# choose condition
cond = cond_smallMR #condG1  # cond_smallMR
#cond = condG1

fig3,ax3 = plt.subplots()
MRindexedimages2Dplot = copy.copy(MRindexedimages2D)
MRindexedimages2Dplot = ma.masked_where(cond, MRindexedimages2Dplot)
    
pp3= ax3.imshow(MRindexedimages2Dplot,origin='lower')
ax3.set_title(txt+f'\nMatching Rate higher than {MinMRthreshold} % ')
cb2 = plt.colorbar(pp3)

GT.format_getimageindex_imshow.__defaults__=(invmapdims,)
ax3.format_coord = GT.format_getimageindex_imshow
ax3.set_xlabel(motorfastaxis)
ax3.set_ylabel(motorslowaxis)

In [ ]:
ma.masked_array(Nbindexedimages2Dplot)

In [ ]:
# True indexed, False non indexed
maskindexed2D

In [ ]:
# True, to be masked, False to be kept
np.invert(maskindexed2D)

# COLLECT .fit files results 

**multiprocessing collection, read .fit files  and  plot results**

In [ ]:
import LaueTools.fitfilereader  as fitreader
from LaueTools.fitfilereader import parsed_fitfile, parsed_fitfileseries
from LaueTools.LaueDataResultsPlots import (StrainMap, EulerAngles,EulerAngles2D,
                                            LatticeParamsMap, PlotNumberIndexedSpots,
                                           PlotMeanDevPixel, StrainMapHistogram)

def getimageindex_fromxyech(xech, yech, invmapdims, xech_stepsize=1., yech_stepsize = 1., startingindex = 0):
    """get imageindex from relative positions xech and yech  (mapdims =(xech dim, yech dim))
    
    invmapdims = dimensions (slow, fast)  axis
    """
    ii = int(xech/xech_stepsize)
    jj = int(yech/yech_stepsize)
    return startingindex + invmapdims[1]*jj+ii

def plotmask(mask, invmapdims, origin="lower", **kwargs):
    straincmap= mpl.colormaps['seismic']
    straincmap.set_over('red')
    straincmap.set_bad('lightgray')
    straincmap.set_under('green')
    fig, ax = plt.subplots()
    ax.imshow(mask, origin=origin, cmap=straincmap, **kwargs)

In [ ]:
!ls -alptr {workdir}

In [ ]:
workdir

In [ ]:
print('Reading .fit files in')

#fitfilefolder = Path(maindatafolder+'/'+subfolderscandata+'//'+'fitfiles_notebook')
fitfilefolder = Path(maindatafolder+'/'+subfolderscandata+'//'+'fitfiles_JSM_Cr_fast')
# fitfilefolder = Path(workdir+'//'+'fitfiles__2grainZr')
# fitfilefolder = Path(workdir+'//'+'fitfiles_notebook_JSM_Zr')
fitfilefolder = Path(workdir+'//'+'fitfiles__1grainZr')
fitfilefolder = Path(workdir+'//'+'fitfiles_notebook_Zr')
fitfilefolder = Path(workdir+'//'+'fitfiles_1grains_veryfast_Zr_ref')
fitfilefolder = Path(workdir+'//'+'fitfiles_1grains_veryfast_Zr_ref_3grains_veryfast_Cr_ref_3grains_veryfast_Cr_ref')
fitfilefolder = Path(workdir+'//'+'fitfiles_1grains_highseek2_192cpus_Zr_ref')
print(fitfilefolder)
print(mapdims)


ffs0 = parsed_fitfileseries(folderpath=fitfilefolder, nb_cols=mapdims[0], nb_rows=mapdims[1],
                            prefix = prefixfilename, suffix = '_g0.fit', use_multiprocessing = True)
ffs0.fitfilefolder = fitfilefolder

# VISUALIZE

In [ ]:
# depending on scan   dmesh fastmotor .. ..  slowmotor .. ..
if expId in ('a321216',):
    motorfastaxis = 'yech'
    motorslowaxis = 'xech'
elif expId in ('blc15488'):
    motorfastaxis = 'xech'
    motorslowaxis = 'yech'
print('--- MAP dimensions  ----')
print(f'`mapdims` (fast axis, slow axis)',mapdims)
print(f'motor                           ({motorfastaxis}, {motorslowaxis})')

In [ ]:
_, ax = PlotNumberIndexedSpots(indexed_fileseries = ffs0 , size = (9,5), origin='lower')
ax.format_coord = lambda x, y: GT.format_getimageindex_imshow(x, y, invmapdims)

In [ ]:
mycmap = mpl.colormaps['PuRd']
mycmap.set_over('yellow')
mycmap.set_bad('black')
mycmap

In [ ]:
# FILTER & MASK
MinimumNbIndexedSpots = 40  # absolute nb of spots
LargestMeanPixelResidue = .2 # pixel unit


cond_lowNbindexed2D=ffs0.NumberOfIndexedSpots.reshape(invmapdims)<MinimumNbIndexedSpots
cond_largepixeldev = ffs0.MeanDevPixel.reshape(invmapdims)>LargestMeanPixelResidue
combinedconds = np.logical_or(cond_lowNbindexed2D,cond_largepixeldev)  # be careful it s OR !!!

#condgrain = np.invert(maskindexed2D)

#cond = condgrain
#cond = cond_smallMR
cond = combinedconds
#cond = None

maxdisplayedMeanPixDev = .3  # in pixel units

_, ax = PlotMeanDevPixel(indexed_fileseries = ffs0 ,size = (9,5),
                         origin='lower', maskingcondition=cond,
                         norm= mpl.colors.Normalize(vmin=0,vmax=maxdisplayedMeanPixDev),
                        cmap=mycmap)
ax.format_coord = lambda x, y: GT.format_getimageindex_imshow(x, y, invmapdims)

In [ ]:
_, ax = PlotNumberIndexedSpots(indexed_fileseries = ffs0 , size = (9,5), origin='lower', maskingcondition=cond)
ax.format_coord = lambda x, y: GT.format_getimageindex_imshow(x, y, invmapdims)

In [ ]:
#plotmask(condgrain, invmapdims, vmax=.9, vmin=0.1)

In [ ]:
#plotmask(cond_smallMR, invmapdims, vmax=.9, vmin=0.1)

### strain map

In [ ]:
straincmap= mpl.colormaps['seismic']
straincmap.set_over('yellow')
straincmap.set_bad('lightgray')  # 'gray'
straincmap.set_under('green')
straincmap

In [ ]:
xech_stepsize, yech_stepsize = 1., 1.

# visualisation parameters
maxamplitudestrain = 20  # in 10-4 units
strain_frame = 'sample' # 'sample' # 'crystal'
#cond = None#np.invert(maskindexed2D) #maskfromMR #None, combinedconds
#cond = condG1

fig, ax = StrainMap(xech = np.arange(mapdims[0])*xech_stepsize,
                    yech = np.arange(mapdims[1])*yech_stepsize,
                    indexed_fileseries = ffs0 , frame =strain_frame,
                    multiplier = 1e4,
                    scale = 'other', vmin=None, vmax=None,
                    norm= mpl.colors.Normalize(vmin=-maxamplitudestrain,vmax=maxamplitudestrain), # in 10-4 units
                    size = (9,5), cmap=straincmap,
                    maskingcondition=combinedconds)

fig.tight_layout()

###  strain magnitude distribution

In [ ]:
frame = 'sample' # crystal

multiplier = 1e4  # 1e4 for 10-4 unit for strain
fig, ax, fit_params = StrainMapHistogram(indexed_fileseries = ffs0,
                    multiplier = multiplier, maskingcondition=combinedconds.ravel(),#maskfromMR.ravel(),#combinedconds.ravel(),
                    size = (9,5),  frame=frame,
                            bins=np.linspace(-30,20,200), #400
                            fit=True)
[_ax.grid() for _ax in ax.ravel()]
fig.tight_layout()

In [ ]:
GT.printgreen(f'Deviatoric Strain distribution fit results:\ncomponent     mean      std')
print(f'         -- in {1./multiplier:e} units  in {frame} frame--')
if frame=='sample':
    listcomponent =['xx','yy','zz','xy','xz','yz']
elif frame=='crystal':
    listcomponent =['aa','bb','cc','ab','ac','bc']
#fit_params
for kk, comp in enumerate(listcomponent):
    amp, meanval, std = fit_params[kk]
    if kk == 3:
        print('-------')
    if meanval > 1:
        GT.printred(f'{comp}         {meanval:.3f}    {std:.3f}')
    elif meanval < -1:
        GT.pcolor(f'{comp}         {meanval:.3f}    {std:.3f}', 'b')
    else: print(f'{comp}         {meanval:.3f}    {std:.3f}')
    if kk == 5:
        print('-------')
fp = np.array(fit_params)
print(f'Largest absolute value: {np.amax(np.fabs(fp[:,1])):.2f}')
print(f'Largest tensile (positive) value: {np.amax(fp[:,1]):.2f}')
print(f'Largest compressive (negative) value: {np.amin(fp[:,1]):.2f}')

In [ ]:
#fit_params

### orientation map

In [ ]:
phi, theta, psi = EulerAngles2D(xech = np.arange(mapdims[0])*xech_stepsize, yech = np.arange(mapdims[1])*yech_stepsize,
              indexed_fileseries = ffs0 , maskingcondition=None, #vlimits=((244,244.5),(44,46),(76, 76.5)),
              size = (9,5))

mphi = np.nanmean(phi)
mtheta = np.nanmean(theta)
mpsi = np.nanmean(psi)

stdphi = np.nanstd(phi)
stdtheta = np.nanstd(theta)
stdpsi = np.nanstd(psi)

print('Statistics on Euler Angles -------------')
for vm, vstd, q in zip([mphi, mtheta, mpsi],[stdphi, stdtheta, stdpsi],['Phi', 'Theta', 'Psi']):
    print(f'For {q}: mean {vm:.2f} deg     std {vstd:.2f} deg')

In [ ]:
fighisteuler, axhisteuler = plt.subplots(1, 3, figsize=(10,3))
titles = ['Phi', 'Theta', 'Psi']

mphi = 244.2 #np.nanmean(phi)
mtheta = np.nanmean(theta)
mpsi = np.nanmean(psi)

nbbins= 60
deltaphi = 0.2
deltatheta = 0.2
deltapsi = 0.2

rangelist = [(quant-delta, quant-delta) for quant, delta in zip([mphi,mtheta,mpsi],[deltaphi, deltatheta, deltapsi])]

fighisteuler.subplots_adjust(hspace = 0.25, wspace = 0.2)
for kk, (axis, data, tit) in enumerate(zip(axhisteuler, (phi, theta, psi), titles)):
    counts, bins, patches = axis.hist(data.ravel(), range=rangelist[kk], bins=nbbins)
    axis.set_title(tit)
    axis.grid()
    

In [ ]:
i_image=getimageindex_fromxyech(28,39, invmapdims)  #fast axis, slow axis, invmapdims = (slow, fast)
i_image

In [ ]:
print(f'For image #{i_image}, Orientation Matrix UB:\n',ffs0.UB[i_image].tolist())

ffile = parsed_fitfile(os.path.join(fitfilefolder,'img_%04d_g0.fit'%i_image), verbose=0)
ffile.UB, ffile.NumberOfIndexedSpots, ffile.MeanDevPixel

### other orientation map (~ inverse pole figure)

In [ ]:
import LaueTools.orientations as ORI

ub = ffs0.UB[i_image]
ORI.myRGB_3(ub)

In [ ]:
i_image = 1800
ub = ffs0.UB[i_image]
ub

In [ ]:
#UBs2Dmap = ffs0.UB.reshape((mapdims[0],mapdims[1],3,3))
rgbs =[]
for _k  in range(len(ffs0.UB)):
    #print('_k',_k)
    ub = ffs0.UB[_k]
    if np.isnan(ub[0][0]):
        rgbs.append([0,0,0])
    else:
        rgbs.append(ORI.myRGB_3(ub))
rgbs = np.array(rgbs)
rgbs=rgbs.reshape((mapdims[1],mapdims[0],3))

In [ ]:
fig, ax = plt.subplots()
ax.imshow(1-rgbs, origin='lower')
ax.format_coord = lambda x, y: GT.format_getimageindex_imshow(x, y, invmapdims)

In [ ]:
# for the same structure
B0matrix = ffs0.B0[0]
B0matrix
tensorcomponent_ij_list =[(0,0),(0,1),(0,2),
                          (1,0),(1,1),(1,2),
                          (2,0),(2,1),(2,2)]
tensortitles = ['100 // x','100 // y','100 // z',
                '010 // x','010 // y','010 // z',
                '001 // x','001 // y','001 // z']
tensortitles2 = ['a* // x','a* // y','a* // z',
                'b* // x','b* // y','b* // z',
                'c* // x','c* // y','c* // z']
cmp_axes = np.array([[1,0,0],[1,0,0],[1,0,0],[0,1,0],[0,1,0],[0,1,0],[0,0,1],[0,0,1],[0,0,1]])
proj_axes = np.array([[1,0,0],[0,1,0],[0,0,1],[1,0,0],[0,1,0],[0,0,1],[1,0,0],[0,1,0],[0,0,1]])

samplerot = GT.matRot([0,1,0],-40)

cosvals = [[] for _ in range(len((tensorcomponent_ij_list)))]
for _k  in range(len(ffs0.UB)):
    #print('_k',_k)
    ubb0 = np.dot(ffs0.UB[_k],B0matrix)
    for cmp, tensorcomponent_ij in enumerate(tensorcomponent_ij_list):
        j,i =tensorcomponent_ij
        if np.isnan(ubb0[i][j]):
            cosval = 1.  # default
        else:
            qdir = np.dot(ubb0, cmp_axes[cmp])
            nqdir = np.linalg.norm(qdir)
            #print('astar_dir',astar_dir)
            cosval = np.abs(np.dot(qdir/nqdir, samplerot.dot(proj_axes[cmp])))
        
        cosvals[cmp].append(cosval)
tab_cmps = []
for kk in range(len(tensorcomponent_ij_list)):
    tab_cmps.append(np.array(cosvals[kk]).reshape(invmapdims))
        


In [ ]:
ubb0 = np.dot(ffs0.UB[2017],B0matrix)
qdir = np.dot(ubb0, cmp_axes[2])
nqdir = np.linalg.norm(qdir)
cval = np.abs(np.dot(qdir/nqdir, samplerot.dot(proj_axes[2])))
qdir/nqdir, zsample, cval

In [ ]:
i_cmp = 5
fig, ax = plt.subplots()
ax.imshow(tab_cmps[i_cmp], cmap='OrRd', origin='lower')
ax.set_title(tensortitles2[i_cmp])
ax.format_coord = lambda x, y: GT.format_getimageindex_imshow(x, y, invmapdims)
ax.set_xlabel(motorfastaxis)
ax.set_ylabel(motorslowaxis)

In [ ]:
fig, ax = plt.subplots(3,3)
for axidx, data, _title in zip(np.ndindex(ax.shape), tab_cmps, tensortitles2):
    #ax[axidx].imshow(data, cmap='OrRd', origin='lower')
    ax[axidx].imshow(np.abs(data), cmap='seismic', origin='lower')
    ax[axidx].set_title(_title)
    ax[axidx].format_coord = lambda x, y: GT.format_getimageindex_imshow(x, y, invmapdims)
    ax[axidx].set_xlabel(motorfastaxis)
    ax[axidx].set_ylabel(motorslowaxis)

In [ ]:
# TODO compute kernel average Map

In [ ]:
# ref UB matrice
i_ref = 2017
UB0 = ffs0.UB[i_ref]
refUB0 = np.linalg.qr(UB0)[0]

print('UB ref',UB0.tolist())

AngleDev = []
for ii in range(mapdims[0]*mapdims[1]):
    repUB = np.linalg.qr(ffs0.UB[ii])[0]
    AngleDev.append(GT.getRotationAngleFrom2Matrices(refUB0, repUB))


AngleDev = np.array(AngleDev)
AngleDev

In [ ]:
i_image = 1377
np.linalg.qr(ffs0.UB[i_image])[0], ffs0.UB[i_image]

In [ ]:
fig, ax = plt.subplots()
im = ax.imshow(AngleDev.reshape(invmapdims), vmin=0, vmax=.4, cmap = mycmap, origin='lower')
ax.set_title(f'angle deviation (deg) from reference image #{i_ref}')
xref, yref = i_ref%mapdims[0], i_ref // mapdims[0]
circle1 = plt.Circle((xref, yref), 0.5, color='r')
plt.colorbar(im)

ax.add_patch(circle1)
ax.format_coord = lambda x, y: GT.format_getimageindex_imshow(x, y, invmapdims)

In [ ]:
imageindex_1 = 888
imageindex_2 = 1377
imageindex_3 = 714

filename1 = os.path.join(corfilefolder,'img_%04d.cor'%imageindex_1)
filename2 = os.path.join(corfilefolder,'img_%04d.cor'%imageindex_2)
filename3 = os.path.join(corfilefolder,'img_%04d.cor'%imageindex_3)
from LaueTools.LaueDataResultsPlots import PlotPeakPos
ff, aa = PlotPeakPos(filename1, label=f'#{imageindex_1}',frame='angles',showindex=True, marker='*',
                     cmap = plt.cm.inferno, alpha=0.9)
aa.grid(True)

PlotPeakPos(filename2, label=f'image #{imageindex_2}',frame='angles',figax=(ff, aa), facecolors="none", edgecolor='r',alpha=0.7)

PlotPeakPos(filename3, label=f'image #{imageindex_3}',frame='angles',figax=(ff, aa), facecolors="none", marker='h', edgecolor='k',alpha=0.7)

In [ ]:
imageindex_1 = 105

filename1 = os.path.join(corfilefolder,'img_%04d.cor'%imageindex_1)

from LaueTools.LaueDataResultsPlots import PlotPeakPos
ff, aa = PlotPeakPos(filename1, frame='angles',showindex=True, label=f'image {imageindex_1}',alpha=0.5)
aa.grid(True)



# [STUDIES] NUMERICAL AND LAUETOOLS ACCURACY

for a given orientation 

## by sampling the reference structure, we test the refinement convergence

UB orientation matrix is constant

In [ ]:
# create N different strcuture with a fixed orientation matrix

c0 = 4.85
mymaterials = {}
for kk in range(10):
    mymaterials['Zn%d'%kk]= ['Zn%d'%kk, [2.6649, 2.6649, c0+0.03*kk, 90, 90, 120], 'wurtzite']
    
mymaterials = {}
a0,b0,c0,alp0, bet0, gam0 = DictLT.dict_Materials['Zn'][1]
N = 20
deltal= np.random.uniform(-.1,.1,size=(N,2))
deltaangles = np.random.uniform(-.1,.1,size=(N,3))
for kk, deltas in enumerate(zip(deltal,deltaangles)):
    da=0
    db, dc = deltas[0]
    dalp, dbet, dgam = deltas[1]
    mymaterials['Zntest%d'%kk]= ['Zntest%d'%kk, [a0+da, b0+db, c0+dc, 90+dalp, 90+dbet, 120+dgam], 'wurtzite']
    
print('\n-------- mymaterials  dict --------------')
print(mymaterials)

In [ ]:
# General parameters indexation
e_min = 5   # [keV]
e_max = 18  # [keV]  # for indexation only ?? (how to refine with emax larger?)
### -----------arguments of IndexSpotsSet() ------------ ###

# ------------  GuessedUBMatrix ---------------
#Since previousResults is limited to single phase or material.Even better we can test a 

# Index refine parameters
dict_indexrefine = {'AngleTolLUT': 0.2,   # Tolerance angle [deg] to recognise angles in the Angular LUT Database
                    'nlutmax': 5,         # Maximum miller index in the Angular LUT Database
                    # Angular tolerance list for auto links between exp. - theo. spots
                    'list matching tol angles': [1,0.5,0.2,0.1,0.1], 
                    # spots set A:  [0,2,58,34] CAUTION max(A) < NBMAXPROBED
                    'central spots indices': [0,1,2],   
                    #number of most intense spot candidate to have a recognisable distance
                    'NBMAXPROBED': 5, # spots set B alias of [0, ..., NBMAXPROBED-1]
                    'MATCHINGRATE_ANGLE_TOL': 0.5,
                    'MinimumMatchingRate': 0,
                    'MinimumNumberMatches': 20,
                    'UseIntensityWeights': False,
                    'nbSpotsToIndex': 10000, # during refinement max number of pairs exp.-theo. spots
                    'CheckOrientation': None, # os.path.join(working_dir,'UBmat.ubs'),
                    'MATCHINGRATE_THRESHOLD_IAL': 2,
                    'GuessedUBMatrix': previousResults
                    }

# list of minimum matching rates that accept to perform the next refinement step (with less tolerance angle)
# Last term is the final minimum matching rate (in percent) to accept the final refinement.
MatchingRate_List = [2,2,2,2,2,2] 
# len(MatchingRate_List) > len('list matching tol angles')

In [ ]:
imageindex = 547
pathfilecor = os.path.join(corfilefolder,'img_%04d.cor'%imageindex)


t0 = time.time()
# (Re)Initialize object spotsset
UBsol = np.array([[-0.946190704006459, -0.165042674567985,  0.280650321594703],
       [ 0.32573740376703 , -0.402236471801761,  0.850366386984638],
       [-0.027936014406621,  0.900811005080669,  0.431842883341892]])
previousResults = (1,[UBsol],0,0)
res=[]
for kk in range(N):
    # missing info from the dictionary filled manually
    key_material = 'Zntest%d'%kk
    
    dataset = ISS.spotsset()

    # Init the dataset with laue spots of .cor file
    dataset.importdatafromfile(pathfilecor)
    dataset.key_material = key_material
    dataset.emin = e_min
    dataset.emax = e_max
    
    dataset.inhibitindexing = True

    # index the current dataset
    dataset.IndexSpotsSet(None,key_material,e_min,e_max,dict_indexrefine,None,
                                  use_file=0, # if 1 , reinit dataset and first argument must be a path to .cor file
                                  IMM=False,
                                  n_LUT = dict_indexrefine['nlutmax'],
                                  LUT = None,
                                  angletol_list = dict_indexrefine['list matching tol angles'],
                                  nbGrainstoFind = 1, 
                                  previousResults = previousResults,
                                  dirnameout_fitfile = fitfilefolder,
                                  corfilename = pathfilecor,
                                  verbose = 0,
                                  MatchingRate_List = MatchingRate_List,
                                dictmaterials = mymaterials,
                                 choose_UB_MinEulerepresentative=False)
    
    pixelresidues = dataset.pixelresidues
    if pixelresidues is None:
        meanpixel= -1
        pixelresidues = []
    else:
        meanpixdev = np.mean(pixelresidues)

    print(f'\n---- Resulst of indexing image #{imageindex} ----')
    print("initial nb of spots",dataset.nbspots)
    print("dict of UB matrix found",dataset.dict_grain_matrix)
    print('dict of nb of indexed spots and matching rate (%):',dataset.dict_grain_matching_rate)
    print('[FOR CHECK] nb of spots for refinements',len(pixelresidues))
    print('Mean pixel residue',meanpixdev) 
    print('elapsed time %.2f sec  with internal multiprocessing of {nbmaxcpus} cpus'%(time.time()-t0))
    if dataset.dict_grain_matrix is {} or any([v is None for v in dataset.dict_grain_matrix.values()]):
        print('Nothing found ')
    res.append([dataset.dict_grain_devstrain, dataset.dict_grain_latticeparameters, dataset.key_material,
                dataset.dict_grain_matching_rate,
                len(pixelresidues), meanpixdev ])
    

In [ ]:
afinal = []
bfinal = []
cfinal = []
alphafinal = []
betafinal = []
gammafinal = []

latparamsfinal = []
Nindexed_MR = []
Pixdev = []
minPixdev = 100
maxNindexed = 0
for el in res:
    grain_ix = 0
    rr = el[1][grain_ix]
    nbindexed_mr = el[3][grain_ix]
    # by default it s grain 0
    pixdev = el[-1]
    if pixdev<minPixdev:
        minPixdev = pixdev
    if nbindexed_mr[0]>maxNindexed:
        maxNindexed = nbindexed_mr[0]
    
    if rr is not None:
        latparamsfinal.append(rr)
        Nindexed_MR.append(nbindexed_mr)
        Pixdev.append(pixdev)
    
latparamsfinal = np.array(latparamsfinal)
Nindexed_MR = np.array(Nindexed_MR)
Pixdev = np.array(Pixdev)
print('nb of refined solutions', len(latparamsfinal))
print('mean results', np.mean(latparamsfinal, axis=0))
print('std results', np.round(np.std(latparamsfinal, axis=0), decimals=6))
print('max mb indexed spots:',maxNindexed)
print('min pix dev:',minPixdev)
#latparamsfinal,Nindexed_MR

In [ ]:
# select only solutions with the SAME Nb of indexed spots
latparamsfinal_select = []
for kk,el in enumerate(latparamsfinal):
    if Nindexed_MR[kk][0]==maxNindexed:
        latparamsfinal_select.append(el)
    
latparamsfinal_select=np.array(latparamsfinal_select)

In [ ]:
print('on selected refined solutions with %d indexed spots'%maxNindexed)
print('nb of refined solutions', len(latparamsfinal_select))
print('Statistical results\n             a,         b,         c,     alpha,   beta,   gamma')
GT.printgreen(f'mean       {np.round(np.mean(latparamsfinal_select, axis=0), decimals=4)}')
GT.printgreen(f'std results {np.round(np.std(latparamsfinal_select, axis=0), decimals=7)}')
print('max mb indexed spots:',maxNindexed)

## by sampling orientation matrix and reference structure lattice parameters, we test the refinement convergence

In [ ]:
# General parameters indexation
e_min = 5   # [keV]
e_max = 18  # [keV]  # for indexation only ?? (how to refine with emax larger?)
### -----------arguments of IndexSpotsSet() ------------ ###

# ------------  GuessedUBMatrix ---------------
#Since previousResults is limited to single phase or material.Even better we can test a 

# Index refine parameters
dict_indexrefine = {'AngleTolLUT': 0.2,   # Tolerance angle [deg] to recognise angles in the Angular LUT Database
                    'nlutmax': 5,         # Maximum miller index in the Angular LUT Database
                    # Angular tolerance list for auto links between exp. - theo. spots
                    'list matching tol angles': [1,0.5,0.2,0.1,0.1], 
                    # spots set A:  [0,2,58,34] CAUTION max(A) < NBMAXPROBED
                    'central spots indices': [0,1,2],   
                    #number of most intense spot candidate to have a recognisable distance
                    'NBMAXPROBED': 5, # spots set B alias of [0, ..., NBMAXPROBED-1]
                    'MATCHINGRATE_ANGLE_TOL': 0.5,
                    'MinimumMatchingRate': 0,
                    'MinimumNumberMatches': 20,
                    'UseIntensityWeights': False,
                    'nbSpotsToIndex': 10000, # during refinement max number of pairs exp.-theo. spots
                    'CheckOrientation': None, # os.path.join(working_dir,'UBmat.ubs'),
                    'MATCHINGRATE_THRESHOLD_IAL': 2,
                    'GuessedUBMatrix': previousResults
                    }

# list of minimum matching rates that accept to perform the next refinement step (with less tolerance angle)
# Last term is the final minimum matching rate (in percent) to accept the final refinement.
MatchingRate_List = [2,2,2,2,2,2] 
# len(MatchingRate_List) > len('list matching tol angles')

In [ ]:
imageindex = 547
pathfilecor = os.path.join(corfilefolder,'img_%04d.cor'%imageindex)

import random
t0 = time.time()
# (Re)Initialize object spotsset
UBsol = np.array([[-0.946190704006459, -0.165042674567985,  0.280650321594703],
       [ 0.32573740376703 , -0.402236471801761,  0.850366386984638],
       [-0.027936014406621,  0.900811005080669,  0.431842883341892]])

res=[]
# random choice of Zntest
N = 30
for ii in range(N):
    # missing info from the dictionary filled manually
    key_material = random.choice(list(mymaterials.keys()))
    randomaxis = np.random.uniform(-1,1,size=3)
    randomangle = np.random.uniform(-2,2,size=1)
    UBmodified = UBsol.dot(GT.matRot(randomaxis, randomangle))
    previousResults = (1,[UBmodified],0,0)
    
    print(f'\n test {ii}/{N-1} key_material', key_material, f'random angle {randomangle[0]:.2f} deg')
    
    dataset = ISS.spotsset()

    # Init the dataset with laue spots of .cor file
    dataset.importdatafromfile(pathfilecor)
    dataset.key_material = key_material
    dataset.emin = e_min
    dataset.emax = e_max
    
    dataset.inhibitindexing = True

    # index the current dataset
    dataset.IndexSpotsSet(None,key_material,e_min,e_max,dict_indexrefine,None,
                                  use_file=0, # if 1 , reinit dataset and first argument must be a path to .cor file
                                  IMM=False,
                                  n_LUT = dict_indexrefine['nlutmax'],
                                  LUT = None,
                                  angletol_list = dict_indexrefine['list matching tol angles'],
                                  nbGrainstoFind = 1, 
                                  previousResults = previousResults,
                                  dirnameout_fitfile = fitfilefolder,
                                  corfilename = pathfilecor,
                                  verbose = 0,
                                  MatchingRate_List = MatchingRate_List,
                                dictmaterials = mymaterials,
                                 choose_UB_MinEulerepresentative=False)
    
    pixelresidues = dataset.pixelresidues
    if pixelresidues is None:
        meanpixel= -1
        pixelresidues = []
        
    else:
        meanpixdev = np.mean(pixelresidues)
        res.append([dataset.dict_grain_devstrain, dataset.dict_grain_latticeparameters, dataset.key_material,
                dataset.dict_grain_matching_rate,
                len(pixelresidues), meanpixdev ])
        
        if 0:

            print(f'\n---- Resulst of indexing image #{imageindex} ----')
            print("initial nb of spots",dataset.nbspots)
            print("dict of UB matrix found",dataset.dict_grain_matrix)
            print('dict of nb of indexed spots and matching rate (%):',dataset.dict_grain_matching_rate)
            print('[FOR CHECK] nb of spots for refinements',len(pixelresidues))
            print('Mean pixel residue',meanpixdev) 
            print('elapsed time %.2f sec  with internal multiprocessing of {nbmaxcpus} cpus'%(time.time()-t0))
            if dataset.dict_grain_matrix is {} or any([v is None for v in dataset.dict_grain_matrix.values()]):
                print('Nothing found ')
    
    

In [ ]:
afinal = []
bfinal = []
cfinal = []
alphafinal = []
betafinal = []
gammafinal = []

latparamsfinal = []
Nindexed_MR = []
Pixdev = []
minPixdev = 100
maxNindexed = 0
for el in res:
    grain_ix = 0
    rr = el[1][grain_ix]
    nbindexed_mr = el[3][grain_ix]
    # by default it s grain 0
    pixdev = el[-1]
    if pixdev<minPixdev:
        minPixdev = pixdev
    if nbindexed_mr[0]>maxNindexed:
        maxNindexed = nbindexed_mr[0]
    
    if rr is not None:
        latparamsfinal.append(rr)
        Nindexed_MR.append(nbindexed_mr)
        Pixdev.append(pixdev)
    
latparamsfinal = np.array(latparamsfinal)
Nindexed_MR = np.array(Nindexed_MR)
Pixdev = np.array(Pixdev)
print('nb of refined solutions', len(latparamsfinal))
print('mean results', np.mean(latparamsfinal, axis=0))
print('std results', np.round(np.std(latparamsfinal, axis=0), decimals=6))
print('max mb indexed spots:',maxNindexed)
print('min pix dev:',minPixdev)
#latparamsfinal,Nindexed_MR

In [ ]:
# select only solutions with the SAME Nb of indexed spots
latparamsfinal_select = []
for kk,el in enumerate(latparamsfinal):
    if Nindexed_MR[kk][0]==maxNindexed:
        latparamsfinal_select.append(el)
    
latparamsfinal_select=np.array(latparamsfinal_select)

In [ ]:
print('on selected refined solutions with %d indexed spots'%maxNindexed)
print('nb of refined solutions', len(latparamsfinal_select))
print('Statistical results\n             a,         b,         c,     alpha,   beta,   gamma')
GT.printgreen(f'mean       {np.round(np.mean(latparamsfinal_select, axis=0), decimals=4)}')
GT.printgreen(f'std results {np.round(np.std(latparamsfinal_select, axis=0), decimals=7)}')
print('max mb indexed spots:',maxNindexed)

## TODO:   simulate laue pattern for any orientation matrix, and test refinement resolution by sampling initial UB and structure

In [ ]:
nbUBs=1
key_material_0 = 'Zn'
emax = 22
simulfolder = corfilefolder # d['folder']
N= 20

import LaueTools.CrystalParameters as CP
import LaueTools.lauecore as LC
from scipy.spatial.transform import Rotation as R

Rs = R.random(nbUBs,random_state=1234)
quatlist = Rs.as_quat()
UBlist = Rs.as_matrix()


detectorparameters = [79.000,975.00,941.00,0,0]

pixelsize = 0.0734
framedim = (2018,2016)

CCDCalibdict = {'dd': detectorparameters[0],
            'xcen': detectorparameters[1],
            'ycen': detectorparameters[2], 'xbet': detectorparameters[3],
            'xgam': detectorparameters[4], 'xpixelsize': pixelsize, 'ypixelsize': pixelsize,
            'CCDLabel': 'sCMOS', 'framedim': framedim, 'detectordiameter': 165,
            'kf_direction': 'Z>0', 'pixelsize': pixelsize}



l_tth, l_chi, l_miller_ind, l_posx, l_posy, l_E = [],[],[],[],[],[]

k=0
Quats=[]
nbspots=np.zeros(len(quatlist))
listnbspots = [0]
res=[]



Rs = R.random(N,random_state=1234)
quatlist = Rs.as_quat()
UBlist = Rs.as_matrix()

# General parameters indexation
e_min = 5   # [keV]
e_max = emax  # [keV]  # for indexation only ?? (how to refine with emax larger?)
### -----------arguments of IndexSpotsSet() ------------ ###

# ------------  GuessedUBMatrix ---------------
#Since previousResults is limited to single phase or material.Even better we can test a 

# Index refine parameters
dict_indexrefine = {'AngleTolLUT': 0.2,   # Tolerance angle [deg] to recognise angles in the Angular LUT Database
                    'nlutmax': 5,         # Maximum miller index in the Angular LUT Database
                    # Angular tolerance list for auto links between exp. - theo. spots
                    'list matching tol angles': [1,0.5,0.2,0.1,0.1], 
                    # spots set A:  [0,2,58,34] CAUTION max(A) < NBMAXPROBED
                    'central spots indices': [0,1,2],   
                    #number of most intense spot candidate to have a recognisable distance
                    'NBMAXPROBED': 5, # spots set B alias of [0, ..., NBMAXPROBED-1]
                    'MATCHINGRATE_ANGLE_TOL': 0.5,
                    'MinimumMatchingRate': 0,
                    'MinimumNumberMatches': 20,
                    'UseIntensityWeights': False,
                    'nbSpotsToIndex': 10000, # during refinement max number of pairs exp.-theo. spots
                    'CheckOrientation': None, # os.path.join(working_dir,'UBmat.ubs'),
                    'MATCHINGRATE_THRESHOLD_IAL': 2,
                    'GuessedUBMatrix': previousResults
                    }

# list of minimum matching rates that accept to perform the next refinement step (with less tolerance angle)
# Last term is the final minimum matching rate (in percent) to accept the final refinement.
MatchingRate_List = [2,2,2,2,2,2] 
# len(MatchingRate_List) > len('list matching tol angles')

for ii in range(N):
    UBmatrix =UBlist[ii]
    grain = CP.Prepare_Grain(key_material_0, UBmatrix)

    s_tth, s_chi, s_miller_ind, s_posx, s_posy, s_E= LC.SimulateLaue_full_np(grain, 5,emax,detectorparameters,
                                                                             pixelsize=pixelsize,detectordiameter=300,
                                                                             dim=framedim,
                                                                            removeharmonics=1)
    

    corfilename = IOLT.writefile_cor('temp_simul', s_tth, s_chi, s_posx, s_posy, s_E,
                       param=CCDCalibdict, overwrite=1, dirname_output=simulfolder)
    pathfilecor= os.path.join(simulfolder,corfilename)

    
    # missing info from the dictionary filled manually
    key_material = random.choice(list(mymaterials.keys()))
    randomaxis = np.random.uniform(-1,1,size=3)
    randomangle = np.random.uniform(-.5,.5,size=1)
    UBmodified = UBmatrix.dot(GT.matRot(randomaxis, randomangle))
    previousResults = (1,[UBmodified],0,0)
    
    print(f'\n test {ii}/{N-1} key_material', key_material, f'random angle {randomangle[0]:.2f} deg')
    
    dataset = ISS.spotsset()

    # Init the dataset with laue spots of .cor file
    dataset.CCDLabel='sCMOS'
    dataset.importdatafromfile(pathfilecor)
    
    dataset.key_material = key_material
    dataset.emin = e_min
    dataset.emax = e_max
    
    dataset.inhibitindexing = True

    # index the current dataset
    dataset.IndexSpotsSet(None,key_material,e_min,e_max,dict_indexrefine,None,
                                  use_file=0, # if 1 , reinit dataset and first argument must be a path to .cor file
                                  IMM=False,
                                  n_LUT = dict_indexrefine['nlutmax'],
                                  LUT = None,
                                  angletol_list = dict_indexrefine['list matching tol angles'],
                                  nbGrainstoFind = 1, 
                                  previousResults = previousResults,
                                  dirnameout_fitfile = fitfilefolder,
                                  corfilename = pathfilecor,
                                  verbose = 0,
                                  MatchingRate_List = MatchingRate_List,
                                dictmaterials = mymaterials,
                                 choose_UB_MinEulerepresentative=False)
    
    pixelresidues = dataset.pixelresidues
    if pixelresidues is None:
        meanpixel= -1
        pixelresidues = []
        
    else:
        meanpixdev = np.mean(pixelresidues)
        res.append([dataset.dict_grain_devstrain, dataset.dict_grain_latticeparameters, dataset.key_material,
                dataset.dict_grain_matching_rate,
                len(pixelresidues), meanpixdev ])
        
        if 0:

            print(f'\n---- Resulst of indexing image #{imageindex} ----')
            print("initial nb of spots",dataset.nbspots)
            print("dict of UB matrix found",dataset.dict_grain_matrix)
            print('dict of nb of indexed spots and matching rate (%):',dataset.dict_grain_matching_rate)
            print('[FOR CHECK] nb of spots for refinements',len(pixelresidues))
            print('Mean pixel residue',meanpixdev) 
            print('elapsed time %.2f sec  with internal multiprocessing of {nbmaxcpus} cpus'%(time.time()-t0))
            if dataset.dict_grain_matrix is {} or any([v is None for v in dataset.dict_grain_matrix.values()]):
                print('Nothing found ')
    
    

In [ ]:
UBlist#res

In [ ]:
afinal = []
bfinal = []
cfinal = []
alphafinal = []
betafinal = []
gammafinal = []

latparamsfinal = []
Nindexed_MR = []
Pixdev = []
minPixdev = 100
maxNindexed = 0
for el in res:
    grain_ix = 0
    rr = el[1][grain_ix]
    nbindexed_mr = el[3][grain_ix]
    # by default it s grain 0
    pixdev = el[-1]
    if pixdev<minPixdev:
        minPixdev = pixdev
    if nbindexed_mr[0]>maxNindexed:
        maxNindexed = nbindexed_mr[0]
    
    if rr is not None:
        latparamsfinal.append(rr)
        Nindexed_MR.append(nbindexed_mr)
        Pixdev.append(pixdev)
    
latparamsfinal = np.array(latparamsfinal)
Nindexed_MR = np.array(Nindexed_MR)
Pixdev = np.array(Pixdev)
print('nb of refined solutions', len(latparamsfinal))
print('mean results', np.mean(latparamsfinal, axis=0))
print('std results', np.round(np.std(latparamsfinal, axis=0), decimals=6))
print('max mb indexed spots:',maxNindexed)
print('min pix dev:',minPixdev)
#latparamsfinal,Nindexed_MR

In [ ]:
# select only solutions with the SAME Nb of indexed spots
latparamsfinal_select = []
for kk,el in enumerate(latparamsfinal):
    if Nindexed_MR[kk][0]>0.8*maxNindexed:
        latparamsfinal_select.append(el)
    
latparamsfinal_select=np.array(latparamsfinal_select)

In [ ]:
print('on selected refined solutions with %d indexed spots'%maxNindexed)
print('nb of refined solutions', len(latparamsfinal_select))
print('Statistical results\n             a,         b,         c,     alpha,   beta,   gamma')
GT.printgreen(f'mean       {np.round(np.mean(latparamsfinal_select, axis=0), decimals=4)}')
GT.printgreen(f'std results {np.round(np.std(latparamsfinal_select, axis=0), decimals=7)}')
print('max mb indexed spots:',maxNindexed)